# Reproducing Wright and McDannald (2019) from CRCNS pag-1

Paper. Wright KM, McDannald MA (2019). Ventrolateral periaqueductal gray neurons prioritize threat
probability over fear output. eLife 8:e45013, doi:10.7554/eLife.45013. The claims file
(claims/wright-2019.yaml) was written from the version of record (PMC6435320, Europe PMC full
text): main text, Materials and methods and all figure legends.

Dataset. CRCNS pag-1, "Ventrolateral periaqueductal gray single-unit activity during multi-cue
fear discrimination in rats", doi:10.6080/K0R49P0V, data description version 0.5 (6 March 2019).
CRCNS data sets have no versions. The bucket holds one copy, under pag-1/live/ (no pag-1/old/),
so that copy is the one used, recorded as `unversioned`. The file read is data.zip, MD5
5437f01d5071fe55a1baedbb146f56e0 as listed in the data set's checksums.md5 (printed below), so
a later change is detectable. It is opened in place over HTTP range requests and never written
to disk. Inside it, the 245 per-unit files data/mFiles/*.mat hold the spike times, events and
waveforms every claim is computed from. The data set's scripts data/cubeScript.m and
data/analysisScript.m are read for two lines below; its precomputed cube (cubeFile/cube.mat) is
not used.

Method details. Every method comes from the claims file's criteria, or, where they leave a
detail open, from the paper (cited where used, in pag.py's docstrings and below). The Methods
print the full regression input of one Onset neuron. The data check finds that neuron and shows
which readings of the open details reproduce it: 2 s before onset for interval fear, 20 s for
total fear (reading b of detail 1 of the claims file) and 0/0 set to 1 (reading c of detail 2).
The claims file added these readings, and gave separate windows for the two ratios, after this
attempt asked about the table (questions/wright-2019.yaml). That reading is the primary one
here. Every reading of the amended claims file is run beside it, and the claims file's earlier
readings (one window, 2 s or 10 s, for both ratios), which the table rules out, are shown as
"former" for comparison.

Claims (39, all tested):

| Claim | Tested |
|---|---|
| fig1c-total-suppression, fig1c-nonlinear-scaling, fig1d-interval-suppression, fig1s1a-nose-poke-rates, fig1s1b-sr-poke-correlation | tested (88 sessions) |
| fig1e-two-clusters, onset-population, ramping-population, fig1s5-hfr-units | tested (245 units) |
| fig2a to fig2s2, fig3, fig4abc to fig4h (14 claims) | tested (29 Onset, 14 Ramping neurons) |
| fig5a to fig5j (11 claims) | tested |
| fig6ab, fig6c, fig6d, fig6ef | tested |

In [ ]:
import time
import resource

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pag

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
T0 = time.time()
units = pag.load_units()
print(f"{len(units)} units loaded in {time.time() - T0:.0f} s")

## Data check

The data set's description (version 0.5, 6 March 2019) says: 245 single units from 88 sessions
of 6 rats, 32 trials per session (6 danger, 6 uncertainty shock, 10 uncertainty omission, 10
safety), 10 s cues, foot shock 2 s after cue offset, waveforms of 32 points over 0.8 ms. Each
check against the values is stated under its output.

In [ ]:
print("checksums.md5 of the data set:", pag.checksums())
info = pd.DataFrame([dict(file=u["file"], session=u["session"], rat=u["rat"], program=u["program"],
                          unit=u["unit"], n_spikes=len(u["spikes"]), n_pokes=len(u["pokes"]),
                          n_shocks=len(u["shock"]), n_trials=len(u["onsets"]),
                          **{t: int((u["types"] == t).sum()) for t in pag.TYPES})
                     for u in units])
print("units", len(info), "| sessions", info.session.nunique(), "| rats", info.rat.nunique())
print(info.groupby("rat").agg(units=("unit", "size"), sessions=("session", "nunique"),
                              program=("program", lambda s: "".join(sorted(set(s))))).to_string())
print("trials per type over units:", info[list(pag.TYPES)].drop_duplicates().to_dict("records"))
print("event channels in every file:", sorted({tuple(u["channels"]) for u in units}))
print("spike trains sorted in every file:", all(u["spikes_sorted"] for u in units))

Counts agree with the description: 245 units, 88 sessions, 6 rats, and every session has
6/6/10/10 trials once uncertainty trials are split by a shock within 30 s of onset. Three
behaviour programs (a, b, c) assign the cue channels differently (counterbalancing of the
auditory cues, data set's cubeScript.m). The files also hold EVT08 and EVT14, which cubeScript.m
does not use; no claim needs them.

In [ ]:
sess = {}
for u in units:
    sess.setdefault(u["session"], []).append(u)
same_events = all(all(np.array_equal(v[0]["onsets"], w["onsets"]) and np.array_equal(v[0]["pokes"], w["pokes"])
                      for w in v) for v in sess.values())
print("all units of a session share cue and poke times:", same_events)
lat = []
for v in sess.values():
    u = v[0]
    for t, ty in zip(u["onsets"], u["types"]):
        s = u["shock"][(u["shock"] > t) & (u["shock"] <= t + 30)]
        lat.append((ty, len(s), (s[0] - t) if len(s) else np.nan))
lat = pd.DataFrame(lat, columns=["type", "n_shock", "latency"])
print(lat.groupby("type").agg(trials=("n_shock", "size"), shocks=("n_shock", "sum"),
                             lat_min=("latency", "min"), lat_max=("latency", "max")).to_string())
gaps = np.concatenate([np.diff(np.sort(v[0]["onsets"])) for v in sess.values()])
print(f"time between cue onsets: min {gaps.min():.1f} s, median {np.median(gaps):.1f} s, max {gaps.max():.1f} s")
print(f"mean time between cue onsets {gaps.mean():.1f} s = {gaps.mean() / 60:.2f} min; "
      f"ITI from cue offset {(gaps.mean() - 10) / 60:.2f} min")

Shocks fall 12.0 s after onset on every danger and uncertainty shock trial (cue 10 s, shock 2 s
after offset, as described), and never on safety or omission trials. Mean inter-onset time is
not consistent with the "mean ITI of 3.5 min" of the paper and the description: onsets are
132 to 195 s apart (printed above), so no ITI reaches 3.5 min. This is recorded as a finding;
no claim uses the ITI length.

In [ ]:
# duplicate recordings: identical or near-identical spike trains within a session, and across
# sessions identical trains
dups = []
for name, v in sess.items():
    for i in range(len(v)):
        for j in range(i + 1, len(v)):
            a, b = v[i]["spikes"], v[j]["spikes"]
            k = np.searchsorted(b, a)
            near = np.minimum(np.abs(a - b[np.clip(k, 0, len(b) - 1)]), np.abs(a - b[np.clip(k - 1, 0, len(b) - 1)]))
            frac = (near <= 0.0005).mean()
            if frac > 0.5:
                dups.append((name, v[i]["unit"], v[j]["unit"], round(frac, 3), len(a), len(b)))
print("unit pairs in a session sharing more than half their spikes (within 0.5 ms):", dups)
hashes = {}
for u in units:
    hashes.setdefault(hash(u["spikes"].tobytes()), []).append(u["file"])
print("identical spike trains in different files:", [v for v in hashes.values() if len(v) > 1])
date_rat = info.drop_duplicates("session").groupby(["rat", info.session.str[4:10]]).size()
print("sessions sharing a rat and date:", date_rat[date_rat > 1].to_dict())

No file is deposited twice, and no two sessions share a rat and date. Five pairs of units
recorded in the same session share 66 to 90% of the spikes of one of them (within 0.5 ms),
which suggests one neuron sorted on two wires. The paper counts 245 units, so it kept both;
the analysis follows the paper and counts them as given, and the claims that use the
populations say whether any pair falls in them. Recorded as a finding.

In [ ]:
rates = np.array([len(u["spikes"]) / (u["spikes"][-1] - u["spikes"][0]) for u in units])
wav = np.array([u["wave"] for u in units])
span = np.array([u["spikes"][-1] - u["spikes"][0] for u in units]) / 60
print(f"session span covered by spikes: {span.min():.0f} to {span.max():.0f} min (description: 93 min sessions)")
# which part of the session the spike trains hold: spike times relative to the nearest cue onset
rel_lo, rel_hi, n_out, n_all = [], [], 0, 0
for u in units:
    d = u["spikes"][:, None] - u["onsets"][None, :]
    near = (d >= -25) & (d <= 40)
    n_out += int((~near.any(1)).sum())
    n_all += len(u["spikes"])
    r = np.where(near, d, np.nan)
    rel_lo.append(np.nanmin(r))
    rel_hi.append(np.nanmax(r))
gap = np.array([np.diff(u["spikes"]).max() for u in units])
print(f"spikes relative to onset: earliest per unit {min(rel_lo):.2f} to {max(rel_lo):.2f} s, latest per unit "
      f"{min(rel_hi):.2f} to {max(rel_hi):.2f} s; {n_out} of {n_all} spikes outside -25 to +40 s")
print(f"longest gap between spikes per unit: median {np.median(gap):.1f} s, range {gap.min():.1f} to {gap.max():.1f} s")
stored = np.array([pag.interval_rate(u, *pag.STORED_S).mean() for u in units])
pre = np.array([pag.interval_rate(u, -2, 0).mean() for u in units])
print(f"2 s pre-cue rate over spikes / span: median {np.median(pre / rates):.2f}; "
      f"over the rate in the stored windows {pag.STORED_S} s: median {np.median(pre / stored):.2f}")
print(f"spikes / span per unit (not a session rate, see above): {rates.min():.2f} to {rates.max():.1f} spikes/s, median {np.median(rates):.1f}")
print(f"waveform values: {wav.min():.4f} to {wav.max():.4f} (no unit stated; 32 points = 0.8 ms at 40 kHz)")
print("units whose largest deflection is negative:", int((wav[np.arange(len(wav)), np.abs(wav).argmax(1)] < 0).sum()), "of", len(wav))
pk = np.concatenate([pag.counts(v[0]["pokes"], v[0]["onsets"], -20, 0, 20)[:, 0] * 3 for v in sess.values()])
print(f"nose pokes per minute in the 20 s before onset: median {np.median(pk):.0f}, max {pk.max():.0f}")

The waveform has no unit in the files or the description; the amplitude ratio used below is
unit-free. The spike trains do not cover the session. Each unit holds spikes only from 20 s before each cue
onset to 31 to 34 s after it, with gaps of about 141 s between trials; the description does not
say so (findings). Spikes divided by the span of the train are therefore about a third of the
firing rate, and the session rate of reading b of detail 5 is taken over the windows from 20 s
before to 30 s after onset, the part every unit holds. The first and last spikes still span 83 to
107 min, near the 93 min sessions the description gives.

**Shared clock.** Spikes, cues and pokes come from one Plexon file per session (description).
The check below contrasts real cue onsets with onsets moved 60 s earlier, into the ITI: the
cue-onset screen of the Methods finds many more responsive units at the true onsets, and poke
rates fall during real cues but not in the shifted windows. Both streams therefore carry the cue
on the trial table's clock.

In [ ]:
def n_responsive(shift):
    hit = 0
    for u in units:
        v = dict(u, onsets=u["onsets"] + shift)
        hit += pag.screen(v, 0.0, 1.0)
    return hit

for shift in (0.0, -60.0):
    sr = [np.mean(pag.total_sr(dict(v[0], onsets=v[0]["onsets"] + shift), 20.0)[v[0]["types"] == "D"]) for v in sess.values()]
    print(f"onsets shifted {shift:+.0f} s: units passing the onset screen {n_responsive(shift)}, "
          f"mean danger suppression ratio over sessions {np.mean(sr):.2f}")

### The paper's example regression input

The Methods (Single-unit, linear regression) print the full regression input of the first
interval of one Onset neuron: Z firing, interval fear and total fear for its 32 trials. The
claims file leaves open the nose poke baseline (detail 1), the 0/0 ratio (detail 2) and the
firing baseline (detail 4). The cell below searches every unit and window for that table.

In [ ]:
TAB_Z = np.array([2.32, 0.51, 0.81, 1.56, 1.41, 2.92, 0.81, -0.40, 0.21, -0.70, -0.85, -0.70, -0.70, -0.70, -0.70, 0.66,
                  0.21, -0.70, -1.00, 1.11, -0.09, -0.85, -0.70, -0.70, -0.70, -0.40, -0.70, -0.70, 0.21, -0.85, 0.06, -0.70])
TAB_IF = np.array([1, 1, 1, 1, -1, 1, 1, -0.33, 1, 1, 1, -1, 1, -1, 0, 1, 1, 1, 1, 1, -0.6, 1, -0.33, 0, 1, 1, 1, -0.5,
                   1, 1, 1, 1.0])
TAB_TF = np.array([1, 1, 1, 1, 0.73, 1, 1, 0.47, 0.64, 0.67, 1, 0.64, 1, 0.40, 0.23, 1, 1, 0.71, 0.43, 1, 0.56, 1,
                   0.06, 0.27, -0.04, 0.14, 0.54, 0.03, 0.47, 1, 0.20, -0.14])
rows = []
for k, u in enumerate(units):
    for bl in ("trial", "type"):
        rows.append(dict(unit=k, file=u["file"], z_baseline=bl,
                         max_abs_diff=np.abs(pag.interval_z(u, 0, 1, bl) - TAB_Z).max()))
zmatch = pd.DataFrame(rows).sort_values("max_abs_diff").head(4)
print("Z firing, closest units:\n", zmatch.to_string(index=False))
k = int(zmatch.iloc[0].unit)
ex = units[k]
rows = []
for w in (2, 10, 20):
    for zero in ("one", "zero"):
        rows.append(dict(baseline_s=w, zero_over_zero=zero,
                         interval_fear_diff=np.abs(pag.interval_sr(ex, 0, 1, w, zero) - TAB_IF).max(),
                         total_fear_diff=np.abs(pag.total_sr(ex, w, zero) - TAB_TF).max()))
print(f"\nunit {ex['file']}: largest difference from the table by poke baseline window")
print(pd.DataFrame(rows).round(3).to_string(index=False))
b2 = pag.counts(ex["pokes"], ex["onsets"], -2, 0, 2)[:, 0]
i1 = pag.counts(ex["pokes"], ex["onsets"], 0, 1, 1)[:, 0]
zz = np.where((b2 == 0) & (i1 == 0))[0]
print("\ntrials (1-based) with no poke in the 2 s baseline or the first second:", (zz + 1).tolist(),
      "-> table interval fear", TAB_IF[zz].tolist())
zf = pag.zscore(pag.interval_rate(ex, 0, 1) - np.floor(pag.interval_rate(ex, -2, 0)))
print("Z firing with the baseline rate rounded down (cubeScript.m): largest difference", round(np.abs(zf - TAB_Z).max(), 3))
cs = pag.open_zip().read("data/cubeScript.m").decode("latin-1").splitlines()
print("\ndata/cubeScript.m, the data set's script:")
for line in cs:
    if any(key in line for key in ("20-s baseline", "time.cue(iii,2)-20);", "poke.base(iii,1) = ",
                                   "cer.cue(iii,:) = ", "cer.s1.interval(iii,:) = ", "replace NaN with 1")):
        print("   ", line.strip())

One unit (rf03092316, rat rf03) reproduces the table to its two printed decimals. Its Z firing
is the trial's own 2 s baseline subtracted, not rounded (reading a of detail 4). Its interval
fear uses the 2 s before onset as the poke baseline, and its total fear the 20 s before onset,
the window of the data set's cubeScript.m. The script's lines printed above show that it uses
this 20 s baseline (poke.base) for the total ratio (cer.cue) and for the 1 s interval ratios
(cer.s1.interval) alike. Ratios of 0/0 are 1.00 in the table (and in
cubeScript.m). After this attempt's question (questions/wright-2019.yaml) the claims file
amended details 1 and 2: the two ratios have separate windows, with readings (interval fear,
total fear) of (a) 2 s and 10 s, (b) 2 s and 20 s, (c) 10 s and 20 s, and 0/0 is (a) 0, (b) left
out or (c) 1. The table on this unit fits reading b of detail 1 and reading c of detail 2. The
analysis below takes that as its primary reading, with Z firing against the trial's own
baseline, and runs every reading of the amended claims file beside it.

## Behaviour: one value per session

The behavioural claims are per session (88), each counted once whatever the number of units.
Readings of the claims file (details 1 to 3, detail 1 and 2 as amended): total fear over the 10 s
(reading a) or 20 s (readings b and c) before onset; interval fear over the 2 s (readings a and
b) or 10 s (reading c) before onset; 0/0 set to 0, left out or set to 1; the session value the
mean of the per-trial ratios or the ratio of the session's mean rates. The window that no
amended reading names for a ratio is shown, labelled "former" or "not a reading". Uncertainty
shock and omission trials are collapsed in the cue (Methods, Population and single-unit firing
analyses).

In [ ]:
S_UNITS = [v[0] for v in sess.values()]
CUES = {"D": ["D"], "U": ["US", "UO"], "S": ["S"]}
WINDOWS = (2.0, 10.0, 20.0)
# detail 1 of the claims file, as amended after this attempt's question: readings (a) 2 s interval
# fear and 10 s total fear, (b) 2 s and 20 s, (c) 10 s and 20 s. A window that is no reading of
# the amended detail (2 s total fear, 20 s interval fear) is shown as "former" or "not a reading".
TOTAL_READING = {2.0: "former (2/2)", 10.0: "a", 20.0: "b, c"}
INTERVAL_READING = {2.0: "a, b", 10.0: "c", 20.0: "not a reading"}
ZEROS = ("zero", "nan", "one")   # detail 2, readings a, b, c


def session_sr(u, a, b, window, zero, agg):
    """Session suppression ratio per cue for the interval (a, b] of the cue."""
    base = pag.poke_baseline(u, window)
    cue = pag.interval_rate(u, a, b, "pokes")
    out = {}
    for c, ts in CUES.items():
        m = np.isin(u["types"], ts)
        if agg == "trial mean":
            out[c] = np.nanmean(pag.suppression(base[m], cue[m], zero))
        else:
            out[c] = pag.suppression(np.array([base[m].mean()]), np.array([cue[m].mean()]), zero)[0]
    return out


def ttest_rel(x, y):
    ok = ~np.isnan(x) & ~np.isnan(y)
    return stats.ttest_rel(x[ok], y[ok])

### fig1c-total-suppression

> Over the 88 recording sessions, suppression ratio over the whole 10 s cue differs by cue, high
> to danger (0.80), intermediate to uncertainty (0.53) and low to safety (0.03), and each cue
> pair differs.

Criterion: within tolerance (0.80 +- 0.08, 0.53 +- 0.053, 0.03 +- 0.05) under one reading, the
repeated-measures ANOVA of cue at p < 0.05 and both paired t-tests at p < 0.05.

In [ ]:
rows = []
SR_TOTAL = {}
for w in WINDOWS:
    for zero in ZEROS:
        for agg in ("trial mean", "ratio of means"):
            if agg == "ratio of means" and zero != "one":
                continue   # the 0/0 rule matters only when a whole session has no poke
            vals = pd.DataFrame([session_sr(u, 0, 10, w, zero, agg) for u in S_UNITS])
            SR_TOTAL[(w, zero, agg)] = vals
            ok = vals.dropna()
            an = pag.rm_anova(ok[["D", "U", "S"]].to_numpy())["A"]
            t1, t2 = ttest_rel(vals.D.values, vals.U.values), ttest_rel(vals.U.values, vals.S.values)
            within = (abs(vals.D.mean() - 0.80) <= 0.08) & (abs(vals.U.mean() - 0.53) <= 0.053) & (abs(vals.S.mean() - 0.03) <= 0.05)
            tests = (an[3] < 0.05) & (t1.pvalue < 0.05) & (t1.statistic > 0) & (t2.pvalue < 0.05) & (t2.statistic > 0)
            rows.append(dict(baseline_s=w, detail1=TOTAL_READING[w], zero=zero, session=agg, D=vals.D.mean(), U=vals.U.mean(), S=vals.S.mean(),
                             F=an[0], df=f"{an[1]},{an[2]}", p_anova=an[3], t_DU=t1.statistic, p_DU=t1.pvalue,
                             t_US=t2.statistic, p_US=t2.pvalue, means_ok=within, tests_ok=tests))
fig1c = pd.DataFrame(rows)
print(fig1c.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("\nsessions:", len(S_UNITS))
ok_rows = fig1c[fig1c.detail1 != TOTAL_READING[2.0]]
print(f"\nreadings of the amended claims file: {len(ok_rows)}; all means within tolerance in {int(ok_rows.means_ok.sum())}, "
      f"tests hold in {int(ok_rows.tests_ok.sum())}")
first = SR_TOTAL[(10.0, "zero", "trial mean")]
print("first reading the criterion lists (10 s total fear, 0/0 = 0, trial mean): "
      + ", ".join(f"{c} {first[c].mean():.3f}" for c in "DUS"))
prim = SR_TOTAL[(20.0, "one", "trial mean")]
print("\nreading b of detail 1 with 0/0 = 1 (the windows that reproduce the Methods table), trial mean, by rat:")
print(prim.assign(rat=[u["rat"] for u in S_UNITS]).groupby("rat").agg(["mean", "size"]).round(2).to_string())

**Result: reproduced.** All eight readings of the amended claims file (total fear over 10 s or 20 s) put the three means within tolerance, and their tests hold. Under the first reading the criterion lists (10 s, 0/0 = 0, mean of trial ratios), danger is 0.768, uncertainty 0.502 and safety -0.002. With the 20 s baseline and 0/0 = 1, the reading of the paper's table, they are 0.799, 0.531 and 0.030, the cue effect is F(2,174) = 595 (p = 1.6e-78), danger against uncertainty t = 12.3 (p = 9.1e-21) and uncertainty against safety t = 21.0 (p = 7.8e-36). The paper reports F = 592.00, t = 12.36 and t = 20.85. The former 2 s readings, which the amended claims file no longer lists, give danger 0.35 to 0.75. Danger is also within tolerance of the legend's 0.83 under every reading (0.768 to 0.799).

### fig1c-nonlinear-scaling

> Suppression scales non-linearly with shock probability. Uncertainty's shock probability is
> closer to safety, but its mean suppression ratio is closer to danger.

Criterion: the uncertainty mean exceeds the midpoint of the danger and safety means, under the
reading fig1c-total-suppression matched on (every reading if none).

In [ ]:
fig1c["midpoint"] = (fig1c.D + fig1c.S) / 2
fig1c["U_above_midpoint"] = fig1c.U > fig1c.midpoint
print(fig1c[["baseline_s", "detail1", "zero", "session", "D", "U", "S", "midpoint", "U_above_midpoint", "means_ok"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

**Result: reproduced.** fig1c-total-suppression matched under all eight readings of the amended claims file, and under each the uncertainty mean is above the midpoint of danger and safety (first reading 0.502 against 0.383; 20 s with 0/0 = 1, 0.531 against 0.415).

### fig1d-interval-suppression

> Suppression ratios in each 1 s cue interval discriminate every cue pair in every interval,
> with a cue effect and a cue x interval interaction.

Criterion: two-way repeated-measures ANOVA (cue x interval) with cue and interaction at p <
0.05, and in all ten intervals danger > uncertainty and uncertainty > safety by paired t-tests
at p < 0.005; partially reproduced with at least eight intervals each. Interval fear uses the 2 s
baseline (readings a and b of detail 1) or 10 s (reading c); 20 s, which no reading names for
interval fear, is shown too.

In [ ]:
rows = []
for w in WINDOWS:
    for zero in ZEROS:
        for agg in ("trial mean", "ratio of means"):
            if agg == "ratio of means" and zero != "one":
                continue
            Y = np.array([[[session_sr(u, k, k + 1, w, zero, agg)[c] for k in range(10)] for c in "DUS"] for u in S_UNITS])
            ok = ~np.isnan(Y).any((1, 2))
            an = pag.rm_anova(Y[ok], ("cue", "interval"))
            nDU = nUS = 0
            tmin = [np.inf, np.inf]
            for k in range(10):
                a, b = ttest_rel(Y[:, 0, k], Y[:, 1, k]), ttest_rel(Y[:, 1, k], Y[:, 2, k])
                nDU += (a.pvalue < 0.005) & (a.statistic > 0)
                nUS += (b.pvalue < 0.005) & (b.statistic > 0)
                tmin = [min(tmin[0], a.statistic), min(tmin[1], b.statistic)]
            rows.append(dict(baseline_s=w, detail1=INTERVAL_READING[w], zero=zero, session=agg, n_sessions=int(ok.sum()),
                             F_cue=an["cue"][0], df_cue=f"{an['cue'][1]},{an['cue'][2]}", p_cue=an["cue"][3],
                             F_int=an["cuexinterval"][0], df_int=f"{an['cuexinterval'][1]},{an['cuexinterval'][2]}",
                             p_int=an["cuexinterval"][3], DU_sig_of_10=nDU, US_sig_of_10=nUS,
                             min_t_DU=tmin[0], min_t_US=tmin[1]))
fig1d = pd.DataFrame(rows)
print(fig1d.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
rd = fig1d[fig1d.detail1 != INTERVAL_READING[20.0]]
full = (rd.p_cue < 0.05) & (rd.p_int < 0.05) & (rd.DU_sig_of_10 == 10) & (rd.US_sig_of_10 == 10)
part = (rd.p_cue < 0.05) & (rd.p_int < 0.05) & (rd.DU_sig_of_10 >= 8) & (rd.US_sig_of_10 >= 8)
print(f"\nreadings of the amended claims file: {len(rd)}; reproduced under {int(full.sum())}, "
      f"partially reproduced under {int((part & ~full).sum())}")

**Result: reproduced, resting on an assumed 0/0 rule (assumed).** With interval fear over the 2 s baseline (readings a and b of detail 1), 0/0 = 1 (reading c of detail 2) and trial means, the cue effect is F(2,174) = 493, p = 2.1e-72, and the cue x interval interaction F(18,1566) = 6.15, p = 9.5e-15. Danger exceeds uncertainty and uncertainty exceeds safety at p < 0.005 in 10 of 10 intervals each (smallest t 3.03 and 6.2). The paper reports F = 489.40, F = 6.21 and all t > 3.00. Of the eight readings of the amended claims file, the four with 0/0 = 1 (2 s or 10 s, trial mean or ratio of means) give 10 of 10 in both comparisons. The four with 0/0 set to 0 or left out give 8 or 9 danger-uncertainty intervals (partially reproduced there). All eight give 10 of 10 uncertainty-safety intervals and both ANOVA effects.

### fig1s1a-nose-poke-rates

> Baseline nose poke rates do not differ between trial types, and poking falls from baseline to
> cue for danger and uncertainty but not for safety.

Criterion: per session and trial type (four), mean baseline and cue poke rates. Baseline
against cue by paired t-test at p < 0.05 (danger and both uncertainty types fall, safety does
not); baseline across the four types by one-way repeated-measures ANOVA at p < 0.05 (no
difference), pairwise t-tests as a variant.

In [ ]:
for w in WINDOWS:
    B = np.array([[pag.poke_baseline(u, w)[u["types"] == t].mean() * 60 for t in pag.TYPES] for u in S_UNITS])
    C = np.array([[pag.interval_rate(u, 0, 10, "pokes")[u["types"] == t].mean() * 60 for t in pag.TYPES] for u in S_UNITS])
    an = pag.rm_anova(B)["A"]
    print(f"baseline {w:.0f} s (detail 1 reading {TOTAL_READING[w]}): baseline across types F({an[1]},{an[2]}) = {an[0]:.2f}, p = {an[3]:.3g}; "
          "mean baseline (pokes/min) " + ", ".join(f"{t} {B[:, i].mean():.1f}" for i, t in enumerate(pag.TYPES)))
    for i, t in enumerate(pag.TYPES):
        r = stats.ttest_rel(B[:, i], C[:, i])
        print(f"   {t:2s}: baseline {B[:, i].mean():5.1f} -> cue {C[:, i].mean():5.1f} pokes/min, t(87) = {r.statistic:6.2f}, p = {r.pvalue:.3g}")
    pw = [(pag.TYPES[i], pag.TYPES[j], stats.ttest_rel(B[:, i], B[:, j]).pvalue) for i in range(4) for j in range(i + 1, 4)]
    print("   variant, baseline pairs p:", ", ".join(f"{a}-{b} {p:.3g}" for a, b, p in pw))

**Result: partially reproduced, resting on an assumed baseline window (assumed).** The supplement's baseline is the total fear window (detail 1 as amended): 10 s under reading a, 20 s under readings b and c. Under every window shown (2, 10 and 20 s), poking falls from baseline to cue for danger and both uncertainty types (t(87) from 10.6 to 17.6, p < 1e-16) and not for safety (p 0.24 to 0.60). Baseline rates across the four trial types do not differ with the 10 s window (F(3,261) = 2.43, p = 0.066), so the claim is reproduced under reading a. They do differ with the 20 s window of readings b and c (F = 3.47, p = 0.017), the window of the paper's table, adopted here. Its means are 39.6, 37.0, 39.1 and 37.8 pokes/min. The former 2 s window gives F = 2.20, p = 0.088.

### fig1s1b-sr-poke-correlation

> Suppression ratio and cue nose poke rate are significantly correlated across sessions,
> particularly for danger and uncertainty.

Criterion: Pearson correlation over the 88 sessions of total cue suppression ratio and cue poke
rate, per cue (reading a) or over all 264 points (reading b); reproduced when, under one
reading, danger and uncertainty correlate negatively at p < 0.05 (and the pooled one under b).

In [ ]:
rows = []
CR = {c: np.array([pag.interval_rate(u, 0, 10, "pokes")[np.isin(u["types"], ts)].mean() * 60 for u in S_UNITS])
      for c, ts in CUES.items()}
for (w, zero, agg), vals in SR_TOTAL.items():
    r = {}
    for c in "DUS":
        ok = ~np.isnan(vals[c].values)
        r[c] = stats.pearsonr(vals[c].values[ok], CR[c][ok])
    allx = np.concatenate([vals[c].values for c in "DUS"])
    ally = np.concatenate([CR[c] for c in "DUS"])
    ok = ~np.isnan(allx)
    rp = stats.pearsonr(allx[ok], ally[ok])
    rows.append(dict(baseline_s=w, detail1=TOTAL_READING[w], zero=zero, session=agg, **{f"r_{c}": r[c][0] for c in "DUS"},
                     **{f"p_{c}": r[c][1] for c in "DUS"}, r_all=rp[0], p_all=rp[1]))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))

**Result: reproduced.** Under every reading of the amended claims file, the total cue suppression ratio and the cue poke rate correlate negatively over the 88 sessions for danger (r -0.60 to -0.73) and uncertainty (r -0.67 to -0.75), all p < 1e-9. Pooled over 264 points, r is -0.76 to -0.79. Safety correlates more weakly (r -0.30 to -0.35, p 0.001 to 0.005). The former 2 s readings also give negative danger and uncertainty correlations at p < 0.001.

## Clusters and populations

### fig1e-two-clusters

> K-means clustering (k = 2) of all 245 neurons on baseline firing rate, waveform half duration
> and amplitude ratio splits them on baseline firing rate alone into a low firing rate cluster
> of 199 and a high firing rate cluster of 46.

Criterion: under one reading of detail 5 (baseline rate: 2 s pre-cue over the 32 trials, or
the session mean; features raw or Z-scored), both cluster sizes within tolerance (199 +- 19.9,
46 +- 4.6), the ANOVA of baseline rate by cluster at p < 0.001, and those of amplitude ratio and
duration at p > 0.05. Initialisation: k-means++ starts, 20 replicates, fixed seed (a standard
choice). The waveform features follow the Methods; how the peaks are picked is in pag.py.

In [ ]:
rng_clu = 0
CLUSTERS = {}
rows = []
for base in ("pre-cue", "session"):
    X = pag.cluster_features(units, base)
    for scale in (False, True):
        lfr = pag.kmeans2(X, scale, seed=rng_clu)
        CLUSTERS[(base, scale)] = lfr
        f = [stats.f_oneway(X[lfr, i], X[~lfr, i]) for i in range(3)]
        rows.append(dict(baseline=base, scaled=scale, LFR=int(lfr.sum()), HFR=int((~lfr).sum()),
                         F_rate=f[0].statistic, p_rate=f[0].pvalue, F_halfdur=f[1].statistic, p_halfdur=f[1].pvalue,
                         F_ampratio=f[2].statistic, p_ampratio=f[2].pvalue,
                         rate_LFR=X[lfr, 0].mean(), rate_HFR=X[~lfr, 0].mean()))
fig1e = pd.DataFrame(rows)
print(fig1e.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
X = pag.cluster_features(units, "pre-cue")
print(f"\nfeature ranges: baseline rate {X[:, 0].min():.2f} to {X[:, 0].max():.1f} spikes/s; half duration "
      f"{X[:, 1].min():.3f} to {X[:, 1].max():.3f} ms; amplitude ratio {X[:, 2].min():.2f} to {X[:, 2].max():.2f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
lfr = CLUSTERS[("pre-cue", False)]
ax[0].scatter(X[:, 1], X[:, 0], c=np.where(lfr, "k", "r"), s=8)
ax[0].set(xlabel="half duration (ms)", ylabel="baseline rate (spikes/s)")
ax[1].scatter(X[:, 2], X[:, 0], c=np.where(lfr, "k", "r"), s=8)
ax[1].set(xlabel="amplitude ratio")
plt.tight_layout()
plt.show()
print("all points shown; red = HFR cluster")

**Result: reproduced.** With the 2 s pre-cue baseline rate and raw features, k-means gives 197 LFR and 48 HFR neurons. The baseline rate differs by cluster (F(1,243) = 835, p = 1.4e-80; paper 829), and half duration (p = 0.97) and amplitude ratio (p = 0.51) do not. Z-scored features give 198 and 47. The session rate (reading b of detail 5), taken over the stored windows from 20 s before to 30 s after each onset because the files hold no spikes between trials (data check), gives 194 and 51, which puts HFR outside tolerance; its cluster mean rates (14.3 and 89.2 spikes/s) are close to those of the pre-cue rate (14.9 and 92.8).

### onset-population

> 29 neurons (from 5 of 6 rats, about 12% of all) increase firing in the first 1 s of danger,
> uncertainty or safety over the 2 s pre-cue baseline, and all of them are in the LFR cluster.

Criterion: the Methods screen (paired two-tailed t-test over a cue's trials, raw rate in the 2 s
before onset against the first 1 s, each cue, p < 0.017, cue higher), uncertainty collapsed;
count 29 +- 2.9, five rats, all in the LFR cluster of fig1e (reading it matched on).

In [ ]:
ONSET = [k for k, u in enumerate(units) if pag.screen(u, 0.0, 1.0)]
print("Onset neurons:", len(ONSET), "| rats:", sorted({units[k]["rat"] for k in ONSET}))
print("per rat:", pd.Series([units[k]["rat"] for k in ONSET]).value_counts().sort_index().to_dict())
for key, lfr in CLUSTERS.items():
    print(f"in LFR cluster under reading {key}: {int(lfr[ONSET].sum())} of {len(ONSET)}")
onset_floor = [k for k, u in enumerate(units) if pag.screen(u, 0.0, 1.0, floor_baseline=True)]
print("variant, baseline rate rounded down as in cubeScript.m:", len(onset_floor), "neurons from",
      len({units[k]["rat"] for k in onset_floor}), "rats")
DUP_UNITS = {(d[0], d[1]) for d in dups} | {(d[0], d[2]) for d in dups}
print("Onset neurons in a pair sharing spikes (data check):",
      [units[k]["file"] for k in ONSET if (units[k]["session"], units[k]["unit"]) in DUP_UNITS])
Xpre = pag.cluster_features(units, "pre-cue")
wf = pd.DataFrame({"file": [units[k]["file"] for k in ONSET], "baseline_rate": Xpre[ONSET, 0],
                   "half_duration_ms": Xpre[ONSET, 1], "amplitude_ratio": Xpre[ONSET, 2]})
print("\nwaveform features of the Onset neurons (for comparison with Figure 1, figure supplement 4):")
print(wf.round(3).to_string(index=False))


**Result: reproduced.** The Methods screen finds 29 Onset neurons from 5 rats (dr01 10, dr02 4, rf02 4, rf03 10, rf08 1), all 29 in the LFR cluster under every clustering reading. The data set script's rounded-down baseline would give 31 from 6 rats. One Onset neuron (dr02092716 SPK04a) shares 66% of its spikes with another unit of its session. The Onset neurons' half durations run from 0.038 to 0.200 ms and their amplitude ratios from 0.13 to 1.00 (table above); the paper gives these per neuron only in a figure, so they are printed for comparison and not tested.

### ramping-population

> Of the remaining neurons, 17 increase firing over baseline in the last 1 s of at least one
> cue; three are in the HFR cluster, and the other 14 (from 4 of 6 rats, about 6%) are the
> Ramping population.

Criterion: the same test on the last 1 s, on the neurons that are not Onset neurons; counts 17
+- 1.7, 3 +- 0.3, 14 +- 1.4, and four rats.

In [ ]:
LATE = [k for k, u in enumerate(units) if k not in ONSET and pag.screen(u, 9.0, 10.0)]
print("late-responsive neurons:", len(LATE))
for key, lfr in CLUSTERS.items():
    r = [k for k in LATE if lfr[k]]
    print(f"cluster reading {key}: HFR {len(LATE) - len(r)}, LFR (Ramping) {len(r)} from rats {sorted({units[k]['rat'] for k in r})}")
LFR = CLUSTERS[("pre-cue", False)]
RAMP = [k for k in LATE if LFR[k]]
HFR_LATE = [k for k in LATE if not LFR[k]]
print("Ramping per rat:", pd.Series([units[k]["rat"] for k in RAMP]).value_counts().sort_index().to_dict())
late_floor = [k for k, u in enumerate(units) if k not in onset_floor and pag.screen(u, 9.0, 10.0, floor_baseline=True)]
print("variant, baseline rounded down:", len(late_floor), "late-responsive")
print("Ramping neurons in a pair sharing spikes:",
      [units[k]["file"] for k in RAMP if (units[k]["session"], units[k]["unit"]) in DUP_UNITS])

**Result: reproduced.** Of the other neurons, 17 pass the screen on the last 1 s. Three are in the HFR cluster under every clustering reading, and the other 14 Ramping neurons come from 4 rats (dr01 4, dr02 2, rf02 1, rf03 7).

## Population firing during the cue

Z-scored 100 ms firing (Methods, Z-score normalization, paragraph 1): 320 bins from 10 s before
onset to 12 s after offset; bin i covers (-10 + 0.1 i, -10 + 0.1 (i + 1)] s from onset. The
ANOVAs use the 110 bins from 1 s before onset to cue offset (bins 90 to 199). Interval Z
firing per trial (paragraph 2) under readings a (trial's own baseline, the reading the
Methods table fixes) and b (trial type's mean baseline) of detail 4.

In [ ]:
Z100 = {k: pag.population_z100(units[k]) for k in ONSET + RAMP}
CUE_BINS = slice(90, 200)


def zcube(ks, bins=CUE_BINS, cues="DUS"):
    return np.array([[Z100[k][c][bins] for c in cues] for k in ks])


def cue_means(ks, a, b, baseline="trial"):
    """Per neuron, mean interval Z firing over each cue's trials (uncertainty collapsed)."""
    out = []
    for k in ks:
        z = pag.interval_z(units[k], a, b, baseline)
        out.append([z[np.isin(units[k]["types"], ts)].mean() for ts in CUES.values()])
    return pd.DataFrame(out, columns=list("DUS"), index=[units[k]["file"] for k in ks])


def by_rat(ks, df):
    return df.assign(rat=[units[k]["rat"] for k in ks]).groupby("rat").agg(["mean"]).round(2).assign(
        n=pd.Series([units[k]["rat"] for k in ks]).value_counts())


def fmt_anova(res):
    return "; ".join(f"{k}: F({v[1]},{v[2]}) = {v[0]:.2f}, p = {v[3]:.2g}" for k, v in res.items())


def paired(df, a, b):
    r = stats.ttest_rel(df[a], df[b])
    return r.statistic, r.pvalue

### fig2a-onset-anova

> Onset population firing (100 ms bins, 1 s before to the end of the cue) shows main effects
> of cue and bin and a cue x bin interaction.

Criterion: repeated-measures ANOVA over the 29 Onset neurons, both main effects and the
interaction at p < 0.01.

In [ ]:
Y = zcube(ONSET)
print("Onset:", fmt_anova(pag.rm_anova(Y, ("cue", "bin"))))
print(f"Z range plotted: {Y.mean(0).min():.2f} to {Y.mean(0).max():.2f}")
t = np.arange(-1, 10, 0.1) + 0.1
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=False)
for i, (c, col) in enumerate(zip("DUS", "rmb")):
    ax[0].plot(t, Y.mean(0)[i], col, label=c)
ax[0].set(title="Onset (n = %d)" % len(ONSET), xlabel="s from cue onset", ylabel="Z firing")
ax[0].legend()
YR = zcube(RAMP)
for i, (c, col) in enumerate(zip("DUS", "rmb")):
    ax[1].plot(t, YR.mean(0)[i], col)
ax[1].set(title="Ramping (n = %d)" % len(RAMP), xlabel="s from cue onset")
plt.tight_layout()
plt.show()
print(f"Ramping Z range plotted: {YR.mean(0).min():.2f} to {YR.mean(0).max():.2f}; all points within the axes")

**Result: reproduced.** Over the 29 Onset neurons the cue effect is F(2,56) = 8.64 (p = 5.4e-4), the bin effect F(109,3052) = 14.19 (p = 1.8e-197), and the interaction F(218,6104) = 1.89 (p = 1.6e-13; paper 1.94), all p < 0.01.

### fig2b-onset-first-interval

> In the first 1 s cue interval Onset neurons fire more to danger than to uncertainty; firing to
> uncertainty is numerically but not significantly above safety, while an ANOVA over the first
> 5 s of uncertainty and safety shows a cue x bin interaction.

Criterion: danger > uncertainty by paired two-tailed t-test at p < 0.05; uncertainty above
safety in mean with p >= 0.05; the ANOVA of 100 ms data for uncertainty and safety, 60 bins
(1 s before onset to 5 s), with a cue x bin interaction at p < 0.05.

In [ ]:
for bl in ("trial", "type"):
    m = cue_means(ONSET, 0, 1, bl)
    print(f"Z baseline {bl}: means D {m.D.mean():.2f}, U {m.U.mean():.2f}, S {m.S.mean():.2f}; "
          "D vs U t(%d) = %.2f, p = %.3g; U vs S t = %.2f, p = %.3g" % ((len(m) - 1,) + paired(m, "D", "U") + paired(m, "U", "S")))
print("ANOVA U and S, 60 bins:", fmt_anova(pag.rm_anova(zcube(ONSET, slice(90, 150), "US"), ("cue", "bin"))))
print("by rat (trial baseline):\n", by_rat(ONSET, cue_means(ONSET, 0, 1)).to_string())

**Result: reproduced.** In the first interval, danger is above uncertainty (t(28) = 5.05, p = 2.4e-5, trial baseline; paper t = 4.54). Uncertainty is above safety in mean (-0.09 against -0.23) without significance (t = 1.36, p = 0.185; paper 1.37, 0.18). The uncertainty and safety ANOVA over 60 bins gives a cue x bin interaction F(59,1652) = 1.41, p = 0.024 (paper 1.50, p = 0.01). The type-baseline reading gives the same.

### fig2b-onset-last-interval

> Differential firing of Onset neurons is absent in the last 1 s cue interval, for danger
> against uncertainty and uncertainty against safety.

Criterion (absence): paired two-tailed t-tests at p < 0.05 on last-interval Z firing; reproduced
when neither pair differs.

In [ ]:
for bl in ("trial", "type"):
    m = cue_means(ONSET, 9, 10, bl)
    d1, d2 = m.D - m.U, m.U - m.S
    print(f"Z baseline {bl}: means D {m.D.mean():.2f}, U {m.U.mean():.2f}, S {m.S.mean():.2f}; "
          "D vs U t = %.2f, p = %.3g (mean difference %.2f, dz %.2f); U vs S t = %.2f, p = %.3g (mean difference %.2f, dz %.2f)"
          % (*paired(m, "D", "U"), d1.mean(), d1.mean() / d1.std(ddof=1), *paired(m, "U", "S"), d2.mean(), d2.mean() / d2.std(ddof=1)))

**Result: reproduced.** In the last interval neither pair differs: danger against uncertainty t = 1.94, p = 0.062 (difference 0.24 Z, dz 0.36), and uncertainty against safety t = 0.58, p = 0.57. The type baseline gives p = 0.080 and 0.62.

### fig2c-ramping-anova

> Ramping population firing (100 ms bins over the cue) shows main effects of cue and bin and a
> cue x bin interaction, with ramping greatest to danger, intermediate to uncertainty and absent
> to safety.

Criterion: as fig2a over the 14 Ramping neurons, all effects at p < 0.01, and the mean Z firing
of the last 1 s orders danger > uncertainty > safety.

In [ ]:
print("Ramping:", fmt_anova(pag.rm_anova(YR, ("cue", "bin"))))
last = YR[:, :, -10:].mean(2).mean(0)
print("mean Z (100 ms bins) in the last 1 s: D %.2f, U %.2f, S %.2f" % tuple(last))
print("mean Z (100 ms bins) 1 s before onset: D %.2f, U %.2f, S %.2f" % tuple(YR[:, :, :10].mean(2).mean(0)))

**Result: reproduced.** Over the 14 Ramping neurons the cue effect is F(2,26) = 32.10, the bin effect F(109,1417) = 12.29, and the interaction F(218,2834) = 4.29 (paper 4.33), all p < 1e-7. In the last 1 s, mean Z firing is danger 1.93, uncertainty 1.20 and safety -0.10.

### fig2d-ramping-first-interval

> Ramping neurons show no differential firing in the first 1 s cue interval.

Criterion (absence): paired two-tailed t-tests at p < 0.05 on first-interval Z firing.

In [ ]:
for bl in ("trial", "type"):
    m = cue_means(RAMP, 0, 1, bl)
    d1, d2 = m.D - m.U, m.U - m.S
    print(f"Z baseline {bl}: means D {m.D.mean():.2f}, U {m.U.mean():.2f}, S {m.S.mean():.2f}; "
          "D vs U t(%d) = %.2f, p = %.3g (difference %.2f); U vs S t = %.2f, p = %.3g (difference %.2f)"
          % (len(m) - 1, *paired(m, "D", "U"), d1.mean(), *paired(m, "U", "S"), d2.mean()))

**Result: reproduced.** In the first interval neither pair differs: danger against uncertainty t(13) = 0.43, p = 0.67, and uncertainty against safety t = 0.49, p = 0.63. The type baseline gives p = 0.54 and 0.77.

### fig2d-ramping-last-interval

> In the last 1 s cue interval Ramping neurons fire more to danger than to uncertainty and more
> to uncertainty than to safety.

Criterion: both differences significant by paired two-tailed t-tests at p < 0.05 in the stated
direction (readings of detail 4).

In [ ]:
for bl in ("trial", "type"):
    m = cue_means(RAMP, 9, 10, bl)
    print(f"Z baseline {bl}: means D {m.D.mean():.2f}, U {m.U.mean():.2f}, S {m.S.mean():.2f}; "
          "D vs U t(%d) = %.2f, p = %.3g; U vs S t = %.2f, p = %.3g" % (len(m) - 1, *paired(m, "D", "U"), *paired(m, "U", "S")))
print("by rat (trial baseline):\n", by_rat(RAMP, cue_means(RAMP, 9, 10)).to_string())

**Result: reproduced.** In the last interval danger exceeds uncertainty (t(13) = 3.14, p = 0.0078) and uncertainty exceeds safety (t = 8.09, p = 2.0e-6). The paper reports 3.17 and 8.26. The type baseline gives the same.

### fig2e-departure-times

> Ramping population firing departs from safety earlier for danger (2.8 s after cue onset at
> p < 0.05, 5.8 s with Bonferroni correction) than for uncertainty (5.7 s, 6.5 s).

Criterion: 1 s windows from cue onset in 100 ms steps; paired two-tailed t-test over the 14
Ramping neurons of danger (or uncertainty) against safety; departure is the first window with p
< 0.05 (or p < 0.00025). Readings of the measure: (a) mean of the 100 ms Z bins in the window;
(b) interval Z firing of the window. Readings of the time: window start or end. Windows start
at 0.0 to 9.9 s (100 per comparison, 0.05/200).

In [ ]:
STARTS = np.round(np.arange(0, 10, 0.1), 1)


def window_firing(ks, measure):
    """neurons x cue (D, U, S) x window."""
    if measure == "a":
        return np.array([[[Z100[k][c][100 + int(round(s * 10)): 110 + int(round(s * 10))].mean() for s in STARTS]
                          for c in "DUS"] for k in ks])
    out = np.empty((len(ks), 3, len(STARTS)))
    for j, s in enumerate(STARTS):
        m = cue_means(ks, s, s + 1).to_numpy()
        out[:, :, j] = m
    return out


WF = {m: window_firing(RAMP, m) for m in "ab"}
rows, DEP = [], {}
for meas, W in WF.items():
    for ci, c in ((0, "D"), (1, "U")):
        r = stats.ttest_rel(W[:, ci], W[:, 2], axis=0)
        for thr in (0.05, 0.00025):
            hit = np.where(r.pvalue < thr)[0]
            j = hit[0] if len(hit) else None
            DEP[(meas, c, thr)] = j
            rows.append(dict(measure=meas, cue=c, threshold=thr, start=STARTS[j] if j is not None else np.nan,
                             end=STARTS[j] + 1 if j is not None else np.nan,
                             t_at_departure=r.statistic[j] if j is not None else np.nan,
                             windows_below=len(hit)))
dep = pd.DataFrame(rows)
print(dep.to_string(index=False))
EXPECT = {("D", 0.05): 2.8, ("D", 0.00025): 5.8, ("U", 0.05): 5.7, ("U", 0.00025): 6.5}
for meas in "ab":
    for when in ("start", "end"):
        d = dep[dep.measure == meas].set_index(["cue", "threshold"])[when]
        ok = all(abs(d[k] - v) <= 0.1 * v + 1e-9 for k, v in EXPECT.items())
        print(f"measure {meas}, time = window {when}: " + ", ".join(f"{k[0]} p<{k[1]} {d[k]:.1f} s" for k in EXPECT) + f" -> all within 10%: {ok}")
fig, ax = plt.subplots(figsize=(5, 3))
for ci, col in ((0, "r"), (1, "m")):
    ax.semilogy(STARTS, stats.ttest_rel(WF["a"][:, ci], WF["a"][:, 2], axis=0).pvalue, col)
ax.axhline(0.05, ls=":", c="k")
ax.set(xlabel="window start (s from onset)", ylabel="p, measure a")
plt.show()

**Result: partially reproduced.** Danger departs from safety before uncertainty at both thresholds under every reading. No reading puts all four times within tolerance. With window starts and measure b, three of the four are: danger 2.7 s at p < 0.05 (paper 2.8), uncertainty 5.2 s (5.7) and 6.4 s (6.5). The danger time at p < 0.00025 is 4.7 s against 5.8 +- 0.58. Measure a gives 2.7, 4.5, 5.1 and 6.4 s. Window ends put every time 1 s later: 3.7, 5.5, 6.1 and 7.4 s (measure a) and 3.7, 5.7, 6.2 and 7.4 s (measure b). There the danger time at p < 0.00025 is within tolerance, but the danger time at p < 0.05 (3.7 s against 2.8 +- 0.28) and the uncertainty time at p < 0.00025 (7.4 s against 6.5 +- 0.65) are not.

### fig2e-rate-no-difference

> The rate of firing increase from the departure window to the last 1 s window does not differ
> between danger and uncertainty, so the earlier departure, not a faster rise, separates them.

Criterion (absence): per Ramping neuron and cue, (firing in the last 1 s window minus firing in
the cue's departure window at p < 0.05) divided by the time between the two windows; paired
t-test at p < 0.05, under the reading fig2e-departure-times matched on (every reading if none).

In [ ]:
for meas, W in WF.items():
    jd, ju = DEP[(meas, "D", 0.05)], DEP[(meas, "U", 0.05)]
    last = len(STARTS) - 10   # window 9.0 to 10.0 s
    if jd is None or ju is None:
        print(meas, "no departure")
        continue
    rd = (W[:, 0, last] - W[:, 0, jd]) / (STARTS[last] - STARTS[jd])
    ru = (W[:, 1, last] - W[:, 1, ju]) / (STARTS[last] - STARTS[ju])
    r = stats.ttest_rel(rd, ru)
    print(f"measure {meas}: departure D {STARTS[jd]} s, U {STARTS[ju]} s; rate D {rd.mean():.3f} +- {rd.std(ddof=1) / np.sqrt(len(rd)):.3f}, "
          f"U {ru.mean():.3f} +- {ru.std(ddof=1) / np.sqrt(len(ru)):.3f} Z/s; t({len(rd) - 1}) = {r.statistic:.2f}, p = {r.pvalue:.3g}")

**Result: reproduced.** No reading of fig2e-departure-times matched in full, so every reading is shown. The rate of increase from the departure window to the last window does not differ between danger and uncertainty: measure a t(13) = -0.20, p = 0.84; measure b t = -0.49, p = 0.63.

### fig2s1-single-trial

> The population patterns hold trial by trial. Onset population firing in the first 1 s exceeds
> baseline on every individual trial of every cue, and Ramping population firing in the last 1 s
> exceeds baseline on danger and uncertainty trials but not safety trials.

Criterion: per trial position within each type (D 1-6, US 1-6, UO 1-10, S 1-10), the population
mean raw rate in the interval against that of the 2 s before onset. Onset exceeds at all 32
positions; Ramping at all 22 danger and uncertainty positions and at no more than 5 of the 10
safety positions.

In [ ]:
def per_position(ks, a, b):
    rows = []
    for t in pag.TYPES:
        n = int((units[ks[0]]["types"] == t).sum())
        for i in range(n):
            cue = np.mean([pag.interval_rate(units[k], a, b)[units[k]["types"] == t][i] for k in ks])
            base = np.mean([pag.interval_rate(units[k], -2, 0)[units[k]["types"] == t][i] for k in ks])
            rows.append(dict(type=t, position=i + 1, interval=cue, baseline=base, above=cue > base))
    return pd.DataFrame(rows)


po = per_position(ONSET, 0, 1)
pr = per_position(RAMP, 9, 10)
print("Onset, first 1 s: positions above baseline by type", po.groupby("type").above.agg(["sum", "size"]).to_dict("index"))
print("Ramping, last 1 s: positions above baseline by type", pr.groupby("type").above.agg(["sum", "size"]).to_dict("index"))
print("Ramping safety positions (interval - baseline, spikes/s):", np.round((pr.interval - pr.baseline)[pr.type == "S"].values, 2).tolist())
print("Onset smallest margin (spikes/s): %.2f" % (po.interval - po.baseline).min())
on_all = bool(po.above.all())
ra_du = bool(pr[pr.type != "S"].above.all())
n_s = int(pr[pr.type == "S"].above.sum())
print(f"criterion: Onset above baseline at all 32 positions {on_all}; Ramping above at all 22 danger and uncertainty "
      f"positions {ra_du}; Ramping above at {n_s} of 10 safety positions (at most 5 allowed) -> "
      f"{'reproduced' if on_all and ra_du and n_s <= 5 else 'not met in full'}")


**Result: partially reproduced.** Onset first-interval firing exceeds the population baseline at all 32 trial positions (smallest margin 5.3 spikes/s). Ramping last-interval firing exceeds baseline at all 22 danger and uncertainty positions, but also at 6 of the 10 safety positions, one more than the criterion's reading allows, so the criterion is not met in full; the Onset part and the danger and uncertainty part hold. The six margins range from 0.57 to 10.64 spikes/s, and four of them are below 3.4.

### fig2s2-nose-poke-cessation

> Onset and Ramping firing is not driven by nose poke cessation. Across neurons, firing to
> danger (Onset first interval, Ramping last interval) is uncorrelated with firing to the first
> second of an ITI poke cessation and is consistently greater.

Criterion: cessation events are ITI pokes followed by at least 2 s without a poke; -1 is the
1 s interval of the poke, +1 the first no-poke second. Readings of the ITI: (a) outside cue,
delay and shock (onset to 12.5 s after it); (b) also excluding the 10 s after cue offset (onset
to 20 s after it). Readings of the measure: (a) raw rates (danger interval, cessation +1);
(b) differential (danger interval minus its 2 s pre-cue baseline; +1 minus -1). Reproduced when,
under one reading, the Pearson correlation across neurons has p >= 0.05 and a two-sided sign
test finds danger greater at p < 0.05, for both populations. Here the -1 interval is the second
that ends at the poke, (t - 1, t], and +1 is (t, t + 1] (a reading of the legend; the
cessation window (t, t + 2] must hold no poke, and (t - 1, t + 2] must lie in the ITI).

In [ ]:
def cessation_times(u, excl_after):
    p = u["pokes"]
    nxt = np.r_[p[1:], np.inf]
    cand = p[(nxt - p) > 2.0]
    on = u["onsets"]
    ok = np.ones(len(cand), bool)
    for t0 in on:
        ok &= ~((cand + 2.0 > t0) & (cand - 1.0 <= t0 + excl_after))
    ok &= (cand - 1.0 > u["spikes"][0]) & (cand + 2.0 < u["spikes"][-1])
    return cand[ok]


def rate_around(spk, times, a, b):
    idx_a = np.searchsorted(spk, times + a, side="right")
    idx_b = np.searchsorted(spk, times + b, side="right")
    return (idx_b - idx_a) / (b - a)


rows = []
for name, ks, (a, b) in (("Onset", ONSET, (0, 1)), ("Ramping", RAMP, (9, 10))):
    for iti, excl in (("a", 12.5), ("b", 20.0)):
        ev = {k: cessation_times(units[k], excl) for k in ks}
        for meas in "ab":
            dg, cs = [], []
            for k in ks:
                u = units[k]
                d = pag.interval_rate(u, a, b)[u["types"] == "D"]
                c1 = rate_around(u["spikes"], ev[k], 0, 1)
                if meas == "a":
                    dg.append(d.mean())
                    cs.append(c1.mean())
                else:
                    dg.append((d - pag.interval_rate(u, -2, 0)[u["types"] == "D"]).mean())
                    cs.append((c1 - rate_around(u["spikes"], ev[k], -1, 0)).mean())
            dg, cs = np.array(dg), np.array(cs)
            r = stats.pearsonr(dg, cs)
            k_, n_, ps = pag.sign_test(dg, cs)
            rows.append(dict(population=name, iti=iti, measure=meas, events_median=int(np.median([len(e) for e in ev.values()])),
                             r=r[0], R2=r[0] ** 2, p_r=r[1], danger_greater=f"{k_}/{n_}", p_sign=ps,
                             danger_mean=dg.mean(), cessation_mean=cs.mean()))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))

**Result: reproduced.** With differential firing (measure b), firing to danger does not correlate with firing to cessation: Onset R2 = 0.0006, p = 0.90, and Ramping R2 = 0.23, p = 0.080. Danger is greater in 26 of 29 Onset neurons (sign test p = 1.5e-5) and 14 of 14 Ramping neurons (p = 1.2e-4; paper 1.22e-4). This holds under both ITI readings. With raw rates (measure a) the two correlate (R2 0.61 to 0.68, p < 0.001), because a neuron's overall rate sets both.

### fig3-onset-ramping-independent

> In the four sessions with an Onset and a Ramping neuron recorded together, trial by trial
> firing of the two is not negatively related; a significant negative beta appears for one
> interval of one pair only.

Criterion: four pairs; per pair and 1 s cue interval, regression with constant over the 32
trials of the Onset neuron's interval Z firing on the Ramping neuron's. Reproduced when at most
two of the 40 regressions give a negative beta with p < 0.05.

In [ ]:
PAIRS = [(i, j) for i in ONSET for j in RAMP if units[i]["session"] == units[j]["session"]]
print("Onset-Ramping pairs in the same session:", len(PAIRS), [(units[i]["file"], units[j]["unit"]) for i, j in PAIRS])
for bl in ("trial", "type"):
    res = []
    for i, j in PAIRS:
        for k in range(10):
            b, p = pag.ols(pag.interval_z(units[i], k, k + 1, bl), pag.interval_z(units[j], k, k + 1, bl))
            res.append(dict(pair=units[i]["session"], interval=k + 1, beta=b[0], p=p[0]))
    res = pd.DataFrame(res)
    neg = res[(res.beta < 0) & (res.p < 0.05)]
    pos = res[(res.beta > 0) & (res.p < 0.05)]
    print(f"Z baseline {bl}: {len(res)} regressions; negative beta at p < 0.05: {len(neg)} {neg[['pair', 'interval']].values.tolist()}; "
          f"positive at p < 0.05: {len(pos)}; beta range {res.beta.min():.2f} to {res.beta.max():.2f}")
# units near the screen's threshold: smallest p with the cue above baseline, 0.017 <= p < 0.05
rows = []
for k, u in enumerate(units):
    for name, (a, b) in (("onset (0-1 s)", (0, 1)), ("late (9-10 s)", (9, 10))):
        pos = [(p, c) for c, (t, p) in pag.screen_p(u, a, b).items() if t > 0]
        if pos and pag.SCREEN_P <= min(pos)[0] < 0.05:
            p, c = min(pos)
            rows.append(dict(screen=name, file=u["file"], cue=c, p=p, LFR=bool(LFR[k]),
                             onset_in_session=sum(units[i]["session"] == u["session"] for i in ONSET),
                             ramping_in_session=sum(units[i]["session"] == u["session"] for i in RAMP)))
near = pd.DataFrame(rows).sort_values(["screen", "p"])
print("\nunits with 0.017 <= screen p < 0.05 (not in either population):")
print(near.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
extra = near[((near.screen.str.startswith("late")) & near.LFR & (near.onset_in_session > 0))
             | ((near.screen.str.startswith("onset")) & (near.ramping_in_session > 0))]
print(f"near-threshold units that would add an Onset-Ramping pair if they passed: {len(extra)}, "
      f"smallest p {extra.p.min():.4f}")


**Result: partially reproduced.** Only three Onset-Ramping pairs share a session, in two sessions (dr01101616 and dr01102316), against the paper's four pairs in four sessions. None of the 30 interval regressions gives a negative beta at p < 0.05 (betas -0.33 to 0.42), so the independence holds for the pairs found. Nine units outside the populations, with screen p from 0.025 to 0.046, would each add a pair if they passed the screen (five late-responsive LFR units in a session with an Onset neuron, four onset-responsive units in a session with a Ramping neuron), so a fourth pair can come from a small difference in the screen; the data do not say which of them the paper counted.

## Single-unit biases and patterns

### fig4abc-onset-single-unit-biases

> In the first 1 s interval, Onset single units' danger and uncertainty firing are correlated
> and biased to danger; uncertainty and safety firing are correlated but the bias to
> uncertainty is not significant; danger firing is biased to the first over the last interval,
> with no correlation between them.

Criterion: per Onset neuron, mean interval Z firing per cue; Pearson correlation and two-sided
sign test per panel. A: correlation p < 0.05 and a danger bias; B: correlation p < 0.05 and sign
test p >= 0.05; C: a first-interval bias and correlation p >= 0.05. Partially when two of three.

In [ ]:
def bias_panels(ks, a, b, first_last):
    out = {}
    for bl in ("trial", "type"):
        m = cue_means(ks, a, b, bl)
        f = cue_means(ks, 0, 1, bl).D
        l = cue_means(ks, 9, 10, bl).D
        rows = []
        for panel, x, y in (("D vs U", m.D, m.U), ("U vs S", m.U, m.S),
                            ("D first vs last" if first_last == "first" else "D last vs first",
                             f if first_last == "first" else l, l if first_last == "first" else f)):
            r = stats.pearsonr(x, y)
            k_, n_, ps = pag.sign_test(x, y)
            rows.append(dict(z_baseline=bl, panel=panel, r=r[0], R2=r[0] ** 2, p_r=r[1], first_greater=f"{k_}/{n_}", p_sign=ps))
        out[bl] = pd.DataFrame(rows)
    return pd.concat(out.values())


def z_null(ks, a, b, seed, n=2000):
    """Correlations of cue means when each neuron's 32 interval Z values are shuffled over trials.
    Z-scoring over the trials ties the cue means (6 D + 16 U + 10 S = 0), so this is the r the
    constraint alone gives."""
    rng = np.random.default_rng(seed)
    Z = [pag.interval_z(units[k], a, b) for k in ks]
    T = [units[k]["types"] for k in ks]
    rs = []
    for _ in range(n):
        M = []
        for z, t in zip(Z, T):
            zp = rng.permutation(z)
            M.append([zp[np.isin(t, ts)].mean() for ts in CUES.values()])
        M = np.array(M)
        rs.append([stats.pearsonr(M[:, 0], M[:, 1])[0], stats.pearsonr(M[:, 1], M[:, 2])[0]])
    rs = np.array(rs)
    m = cue_means(ks, a, b)
    obs = [stats.pearsonr(m.D, m.U)[0], stats.pearsonr(m.U, m.S)[0]]
    for i, nm in enumerate(("D vs U", "U vs S")):
        lo, med, hi = np.percentile(rs[:, i], [2.5, 50, 97.5])
        print(f"{nm}: observed r {obs[i]:.2f}; trials shuffled within neuron ({n}x): median r {med:.2f}, "
              f"95% range {lo:.2f} to {hi:.2f}, fraction as negative or more {np.mean(rs[:, i] <= obs[i]):.3f}")


print(bias_panels(ONSET, 0, 1, "first").to_string(index=False, float_format=lambda x: f"{x:.3g}"))
z_null(ONSET, 0, 1, seed=11)


**Result: reproduced.** A: danger and uncertainty correlate (r = -0.47, p = 0.010) and 22 of 29 neurons fire more to danger (sign p = 0.008). B: uncertainty and safety correlate (r = -0.55, p = 0.002) and the bias to uncertainty is not significant (17 of 29, p = 0.46). C: 22 of 29 fire more to danger in the first interval than in the last (p = 0.008), with no correlation (r = 0.34, p = 0.070). Both correlations are negative. Interval Z firing has mean zero over each neuron's 32 trials (6 D + 16 U + 10 S = 0), so the cue means of a neuron trade off. With each neuron's trials shuffled, the constraint alone gives a median r of -0.49 for A (95% range -0.73 to -0.15) and -0.68 for B (-0.84 to -0.43); the observed -0.47 and -0.55 are within these ranges. The correlations the criterion asks for are present, but they are no stronger than this procedure gives without any relation between the cues. The type baseline gives the same outcome.

### fig4d-onset-patterns

> The most common pattern of Onset neurons in the first 1 s interval is danger > uncertainty >
> safety (14 neurons), the only pattern with more neurons than expected by chance.

Criterion: each neuron placed by the order of its mean interval Z firing to the three cues;
chance from 1000 shuffles, (a) the three values permuted within each neuron, or (b) each cue's
values permuted across neurons; "above chance" (a) above the 95th percentile of the shuffled
counts, or (b) above Q3 + 1.5 IQR. Judged on the four categories of the Methods (DUS, DSU, SUD,
UDS); all six orders shown as a variant. Count 14 +- 1.4, the largest, the only one above chance.

In [ ]:
ORDERS = ["DUS", "DSU", "SUD", "UDS", "USD", "SDU"]


def order_of(v):
    return "".join(np.array(list("DUS"))[np.argsort(-v, kind="stable")])


def patterns(ks, a, b, seed):
    rng = np.random.default_rng(seed)
    out = []
    for bl in ("trial", "type"):
        M = cue_means(ks, a, b, bl).to_numpy()
        obs = pd.Series([order_of(v) for v in M]).value_counts().reindex(ORDERS, fill_value=0)
        for sh in "ab":
            null = []
            for _ in range(1000):
                if sh == "a":
                    S = np.array([rng.permutation(v) for v in M])
                else:
                    S = np.column_stack([rng.permutation(M[:, c]) for c in range(3)])
                null.append(pd.Series([order_of(v) for v in S]).value_counts().reindex(ORDERS, fill_value=0).values)
            null = np.array(null)
            q1, q3 = np.percentile(null, [25, 75], axis=0)
            p95 = np.percentile(null, 95, axis=0)
            whisk = q3 + 1.5 * (q3 - q1)
            for i, o in enumerate(ORDERS):
                out.append(dict(z_baseline=bl, shuffle=sh, order=o, observed=obs[o], median=np.median(null[:, i]),
                                p95=p95[i], whisker=whisk[i], above_p95=obs[o] > p95[i], above_whisker=obs[o] > whisk[i]))
    return pd.DataFrame(out)


pat_on = patterns(ONSET, 0, 1, seed=4)
print(pat_on.to_string(index=False))


def pattern_verdict(df, target="DUS", four=ORDERS[:4]):
    rows = []
    for (bl, sh), g in df.groupby(["z_baseline", "shuffle"]):
        g4 = g[g.order.isin(four)].set_index("order")
        for rule in ("above_p95", "above_whisker"):
            above = g4.index[g4[rule]].tolist()
            rows.append(dict(z_baseline=bl, shuffle=sh, rule=rule, count=int(g4.loc[target, "observed"]),
                             largest=g4.observed.idxmax() == target and (g4.observed == g4.observed.max()).sum() == 1,
                             above_chance=above))
    return pd.DataFrame(rows)


print(pattern_verdict(pat_on).to_string(index=False))

**Result: reproduced under shuffle reading a.** The danger > uncertainty > safety pattern holds 14 of 29 Onset neurons, the largest of the four categories. With the three values permuted within each neuron it is the only category above the 95th percentile (8.05) and above the whisker (9.0). With each cue permuted across neurons (reading b), the chance median for this order is 15, and only safety > uncertainty > danger (4 against a whisker of 3.5) is above chance.

### fig4efg-ramping-single-unit-biases

> In the last 1 s interval, Ramping single units' danger and uncertainty firing are correlated
> and biased to danger; uncertainty and safety firing are not correlated but are biased to
> uncertainty; danger firing is biased to the last over the first interval, with no correlation
> between them.

Criterion: as fig4abc on the 14 Ramping neurons and the last interval. E: correlation p < 0.05
and a danger bias; F: correlation p >= 0.05 and an uncertainty bias; G: a last-interval bias and
correlation p >= 0.05. Partially when two of three hold.

In [ ]:
print(bias_panels(RAMP, 9, 10, "last").to_string(index=False, float_format=lambda x: f"{x:.3g}"))
z_null(RAMP, 9, 10, seed=12)


**Result: partially reproduced.** E: 12 of 14 Ramping neurons fire more to danger than uncertainty (p = 0.013), but the correlation is not significant (r = -0.45, p = 0.11; type baseline p = 0.17). F: no correlation (p = 0.094) and a bias to uncertainty (14 of 14, p = 1.2e-4). G: 12 of 14 fire more in the last interval than the first (p = 0.013), with no correlation (p = 0.33). Two of three panels hold. As for the Onset neurons, the shuffled-trial correlations are negative (median -0.49 for E, -0.70 for F), and the observed -0.45 and -0.46 are within their 95% ranges.

### fig4h-ramping-patterns

> The most common pattern of Ramping neurons in the last 1 s interval is danger > uncertainty >
> safety (11 neurons), the only pattern above chance.

Criterion: as fig4d, on the 14 Ramping neurons and the last interval; count 11 +- 1.1.

In [ ]:
pat_ra = patterns(RAMP, 9, 10, seed=8)
print(pat_ra.to_string(index=False))
print(pattern_verdict(pat_ra).to_string(index=False))

**Result: reproduced under shuffle reading a.** The danger > uncertainty > safety pattern holds 12 of 14 Ramping neurons, within 11 +- 1.1 and the largest. It is the only category above chance under within-neuron shuffles (95th percentile 5, whisker 6). Under across-neuron shuffles no category is above chance.

### fig1s5-hfr-units

> The three late-responsive neurons of the HFR cluster each fire most to danger in the last 1 s
> cue interval.

Criterion: for each, mean last-interval firing (raw rate, and mean interval Z as a second
reading) higher for danger than for uncertainty and safety; reproduced when this holds for all
three under one reading.

In [ ]:
print("HFR late-responsive neurons:", len(HFR_LATE))
for k in HFR_LATE:
    u = units[k]
    raw = {c: pag.interval_rate(u, 9, 10)[np.isin(u["types"], ts)].mean() for c, ts in CUES.items()}
    z = {c: pag.interval_z(u, 9, 10)[np.isin(u["types"], ts)].mean() for c, ts in CUES.items()}
    print(f"{u['file']} ({u['rat']}): raw D {raw['D']:.1f}, U {raw['U']:.1f}, S {raw['S']:.1f} spikes/s "
          f"-> danger highest {raw['D'] > max(raw['U'], raw['S'])}; Z D {z['D']:.2f}, U {z['U']:.2f}, S {z['S']:.2f} "
          f"-> danger highest {z['D'] > max(z['U'], z['S'])}")

**Result: reproduced.** Each of the three HFR late-responsive neurons fires most to danger in the last interval, by raw rate (141.5, 89.3 and 89.8 spikes/s against at most 120.9, 64.7 and 81.5) and by interval Z.

## Single-unit regressions

Per neuron and interval, OLS with a constant over the 32 trials of interval Z firing on interval
fear, total fear and probability (1.00, 0.375, 0.00) (Methods, Single-unit, linear regression).
Readings of the claims file, details 1 and 2 as amended: the poke baselines (interval fear /
total fear) (a) 2 s / 10 s, (b) 2 s / 20 s, (c) 10 s / 20 s; 0/0 set to 0, left out, or set to 1;
Z firing against the trial's own 2 s baseline or the type's mean (detail 4). That is 18
readings. The primary one is the reading that reproduces the Methods table (data check): 2 s /
20 s, 0/0 = 1, the trial's own baseline. Each claim prints the primary reading in full and, for
every reading, whether its criterion holds. The claims file's earlier readings (one window for
both ratios, 2 s / 2 s or 10 s / 10 s, 12 readings), which the table rules out, are shown below
them as "former" for comparison and do not count towards the outcome.

In [ ]:
# detail 1 of the claims file, as amended after this attempt's question: (interval fear window,
# total fear window), readings a, b, c
POKE_READINGS = {"a 2/10": (2.0, 10.0), "b 2/20": (2.0, 20.0), "c 10/20": (10.0, 20.0)}
# the claims file's earlier readings (one window for both ratios), shown for comparison only
FORMER = {"former 2/2": (2.0, 2.0), "former 10/10": (10.0, 10.0)}
ALL_POKE = {**POKE_READINGS, **FORMER}
ZERO_READINGS = ("zero", "nan", "one")   # detail 2, readings a, b, c
READINGS = [(w, zr, zb) for w in POKE_READINGS for zr in ZERO_READINGS for zb in ("trial", "type")]
FORMER_READINGS = [(w, zr, zb) for w in FORMER for zr in ZERO_READINGS for zb in ("trial", "type")]
PRIMARY = ("b 2/20", "one", "trial")   # the reading that reproduces the Methods table (data check)
PROB = np.array([pag.PROB[t] for t in units[0]["types"]])
CUE_IV = [(k, k + 1.0) for k in range(10)]
POST_IV = [(10 + 0.5 * k, 10.5 + 0.5 * k) for k in range(10) if k != 4]   # 500 ms, shock (12.0-12.5 s) left out
_cache = {}


def regressors(k, reading, iv):
    key = (k, reading, iv)
    if key not in _cache:
        u = units[k]
        w, zr, zb = reading
        wi, wt = ALL_POKE[w]
        _cache[key] = dict(z=pag.interval_z(u, iv[0], iv[1], zb), IF=pag.interval_sr(u, iv[0], iv[1], wi, zr),
                           TF=pag.total_sr(u, wt, zr))
    return _cache[key]


def betas(ks, reading, ivs=CUE_IV, regs=("IF", "TF", "P"), prob=None):
    """neurons x intervals x regressors."""
    pr = PROB if prob is None else prob
    out = np.empty((len(ks), len(ivs), len(regs)))
    for i, k in enumerate(ks):
        for j, iv in enumerate(ivs):
            r = regressors(k, reading, iv)
            X = np.column_stack([pr if g == "P" else r[g] for g in regs])
            out[i, j] = pag.ols(r["z"], X)[0]
    return out


def tuning_prob(pu):
    p = PROB.copy()
    p[PROB == 0.375] = pu
    return p


def onesamp(x):
    return stats.ttest_1samp(x, 0.0)


def show_readings(fn):
    """Run a claim's check under every reading; fn returns a dict with a boolean 'holds'."""
    rows = [dict(poke=r[0], zero=r[1], z=r[2], reading=r in READINGS, **fn(r)) for r in READINGS + FORMER_READINGS]
    df = pd.DataFrame(rows)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
    rd, fm = df[df.reading], df[~df.reading]
    print(f"criterion holds under {int(rd.holds.sum())} of {len(rd)} readings of the amended claims file "
          f"(former same-window readings: {int(fm.holds.sum())} of {len(fm)})")
    return df

### fig5a-onset-probability-first-interval

> In the first 1 s cue interval the Onset population's probability beta is positive and
> significant and exceeds both fear betas, neither of which differs from zero.

Criterion: probability beta exceeds each fear beta (paired two-tailed t-tests, p < 0.05), is
above zero (single-sample t-test, p < 0.005), and neither fear beta differs from zero at p <
0.005; 0.05 shown as a variant.

In [ ]:
def check5a(r, verbose=False):
    B = betas(ONSET, r, CUE_IV[:1])[:, 0]
    IF, TF, P = B.T
    tPT, tPI = stats.ttest_rel(P, TF), stats.ttest_rel(P, IF)
    zP, zT, zI = onesamp(P), onesamp(TF), onesamp(IF)
    if verbose:
        print(f"mean betas: interval fear {IF.mean():.3f}, total fear {TF.mean():.3f}, probability {P.mean():.3f} (n = {len(P)})")
        print(f"P vs TF t(28) = {tPT.statistic:.2f}, p = {tPT.pvalue:.3g}; P vs IF t = {tPI.statistic:.2f}, p = {tPI.pvalue:.3g}")
        print(f"against zero: P t = {zP.statistic:.2f}, p = {zP.pvalue:.3g}; TF t = {zT.statistic:.2f}, p = {zT.pvalue:.3g}; IF t = {zI.statistic:.2f}, p = {zI.pvalue:.3g}")
    holds = (tPT.statistic > 0) & (tPT.pvalue < 0.05) & (tPI.statistic > 0) & (tPI.pvalue < 0.05) & (zP.statistic > 0) & (zP.pvalue < 0.005) \
        & (zT.pvalue >= 0.005) & (zI.pvalue >= 0.005)
    return dict(P=P.mean(), TF=TF.mean(), IF=IF.mean(), p_PvTF=tPT.pvalue, p_PvIF=tPI.pvalue, p_P0=zP.pvalue,
                p_TF0=zT.pvalue, p_IF0=zI.pvalue, holds=holds, holds_at_005=holds & (zT.pvalue >= 0.05) & (zI.pvalue >= 0.05))


check5a(PRIMARY, verbose=True)
r5a = show_readings(check5a)
print("by rat, primary reading (mean betas IF, TF, P):")
print(pd.DataFrame(betas(ONSET, PRIMARY, CUE_IV[:1])[:, 0], columns=["IF", "TF", "P"]).assign(rat=[units[k]["rat"] for k in ONSET]).groupby("rat").agg(["mean", "size"]).round(2).to_string())

**Result: reproduced.** First-interval mean betas over the 29 Onset neurons are probability 0.863, total fear 0.024 and interval fear -0.034. Probability exceeds total fear (t(28) = 3.74, p = 8.5e-4; paper 3.75) and interval fear (t = 4.55, p = 9.6e-5; paper 4.57). Probability is above zero (p = 6.2e-5) and neither fear beta differs from zero (p = 0.79, 0.58). All 18 readings of the amended claims file meet the criterion, also at 0.05, as do the 12 former readings.

### fig5bc-onset-beta-bias

> Onset single-unit betas are positively biased for threat probability but not for total or
> interval fear.

Criterion: first-interval betas of the 29 Onset neurons; two-sided sign test of each against
zero at p < 0.05. Probability biased positive; neither fear regressor (p >= 0.05).

In [ ]:
def check5bc(r):
    B = betas(ONSET, r, CUE_IV[:1])[:, 0]
    s = [pag.sign_test(B[:, i]) for i in range(3)]
    holds = (s[2][2] < 0.05) & (s[2][0] > s[2][1] / 2) & (s[0][2] >= 0.05) & (s[1][2] >= 0.05)
    return dict(IF_pos=f"{s[0][0]}/{s[0][1]}", p_IF=s[0][2], TF_pos=f"{s[1][0]}/{s[1][1]}", p_TF=s[1][2],
                P_pos=f"{s[2][0]}/{s[2][1]}", p_P=s[2][2], holds=holds)


r5bc = show_readings(check5bc)

**Result: reproduced, resting on the primary reading (assumed).** Under the primary reading, 22 of 29 probability betas are positive (sign p = 0.008), while interval fear (14 of 29, p = 1.0) and total fear (17 of 29, p = 0.46) are not biased. 17 of 18 readings of the amended claims file meet the criterion. The exception is reading b of detail 1 with 0/0 trials left out and the trial baseline, where probability is 20 of 29 (p = 0.061). All 12 former readings meet it.

### fig5d-onset-all-intervals

> Over the ten 1 s cue intervals, Onset threat probability signaling is highest in the first
> interval, persists several seconds and diminishes, while total and interval fear account for
> firing in no interval; an ANOVA shows a main effect of regressor and a regressor x interval
> interaction.

Criterion: repeated-measures ANOVA (regressor x interval), both effects at p < 0.01; mean
probability beta largest in the first interval and above zero (p < 0.005) in it and in at least
one later interval; no fear beta differs from zero at p < 0.005 in any interval.

In [ ]:
def check5d(r, verbose=False):
    B = betas(ONSET, r)                       # n x 10 x 3
    an = pag.rm_anova(B.transpose(0, 2, 1), ("regressor", "interval"))
    pP = np.array([onesamp(B[:, j, 2]).pvalue for j in range(10)])
    tP = np.array([onesamp(B[:, j, 2]).statistic for j in range(10)])
    pF = np.array([[onesamp(B[:, j, i]).pvalue for j in range(10)] for i in range(2)])
    mP = B[:, :, 2].mean(0)
    if verbose:
        print(fmt_anova(an))
        print("mean probability beta by interval:", np.round(mP, 3).tolist())
        print("p against zero, probability:", [f"{p:.2g}" for p in pP])
        print("smallest p against zero, interval fear %.3g, total fear %.3g" % (pF[0].min(), pF[1].min()))
    holds = (an["regressor"][3] < 0.01) & (an["regressorxinterval"][3] < 0.01) & (mP.argmax() == 0) \
        & (pP[0] < 0.005) & (tP[0] > 0) & np.any((pP[1:] < 0.005) & (tP[1:] > 0)) & (pF.min() >= 0.005)
    return dict(F_reg=an["regressor"][0], p_reg=an["regressor"][3], F_int=an["regressorxinterval"][0],
                p_int=an["regressorxinterval"][3], peak_interval=int(mP.argmax()) + 1,
                P_sig_intervals=int(((pP < 0.005) & (tP > 0)).sum()), min_p_fear=pF.min(), holds=holds)


check5d(PRIMARY, verbose=True)
r5d = show_readings(check5d)

**Result: reproduced, resting on the primary reading (assumed).** The regressor effect is F(2,56) = 7.16 (p = 0.0017; paper 7.16) and the regressor x interval interaction F(18,504) = 2.35 (p = 0.0014; paper 2.38). The probability beta peaks in interval 1 (0.863) and declines to 0.24, and is above zero at p < 0.005 in intervals 1 to 4. No fear beta differs from zero in any interval (smallest p 0.156). 7 of 18 readings of the amended claims file meet the criterion, all with the trial baseline. The nine type-baseline readings miss by an ANOVA effect at p above 0.01. The other two (0/0 left out, interval fear over 2 s) miss by a peak in interval 2 and a regressor effect at p = 0.015 and 0.028. No reading has a fear beta different from zero at p < 0.005 (smallest p 0.011). Of the former readings, 8 of 12 meet the criterion.

### fig5s1-regressor-combinations

> Onset neurons prioritize probability whatever the combination of regressors; a fear beta
> reaches significance only when total fear is the sole regressor, and even then its relation is
> weaker than that of probability.

Criterion: regressions over the ten intervals with (A) P + TF, (B) P + IF, (C) IF, (D) TF,
(E) P. "Significance for fear output": (a) a single-sample t-test of the mean beta against
zero in any interval at p < 0.005, or (b) the main effect of interval in C to E at p < 0.05.
Reproduced when, under one reading, no fear regressor is significant in A, B and C, total fear
is in D, the D beta is below the E beta ("weaker": (a) first-interval means, (b) the largest
mean over intervals), and A and B each show a main effect of regressor (p < 0.05) with the
probability beta above the fear beta.

In [ ]:
COMBOS = {"A": ("P", "TF"), "B": ("P", "IF"), "C": ("IF",), "D": ("TF",), "E": ("P",)}


def combo_check(ks, r, verbose=False):
    out = {}
    Bs = {c: betas(ks, r, regs=g) for c, g in COMBOS.items()}
    sig_a, sig_b = {}, {}
    for c, g in COMBOS.items():
        for i, reg in enumerate(g):
            ps = np.array([onesamp(Bs[c][:, j, i]).pvalue for j in range(10)])
            sig_a[(c, reg)] = bool((ps < 0.005).any())
        if len(g) == 1:
            an = pag.rm_anova(Bs[c][:, :, 0])["A"]
            sig_b[(c, g[0])] = an[3] < 0.05
            out[f"F_int_{c}"] = an[0]
            out[f"p_int_{c}"] = an[3]
    for c in "AB":
        an = pag.rm_anova(Bs[c].transpose(0, 2, 1), ("regressor", "interval"))
        out[f"p_reg_{c}"] = an["regressor"][3]
        out[f"P_above_{c}"] = Bs[c][:, :, 0].mean() > Bs[c][:, :, 1].mean()
    res = {}
    for rd, sig in (("a", sig_a), ("b", sig_b)):
        fearA = sig_a[("A", "TF")] if rd == "a" else False
        fearB = sig_a[("B", "IF")] if rd == "a" else False
        nofear = not fearA and not fearB and not sig[("C", "IF")]
        dsig = sig[("D", "TF")]
        res[rd] = dict(nofear_ABC=nofear, TF_sig_in_D=dsig)
    weaker = {"a": Bs["D"][:, 0, 0].mean() < Bs["E"][:, 0, 0].mean(),
              "b": Bs["D"][:, :, 0].mean(0).max() < Bs["E"][:, :, 0].mean(0).max()}
    AB = all(out[f"p_reg_{c}"] < 0.05 and out[f"P_above_{c}"] for c in "AB")
    holds = {}
    for rd in "ab":
        for wk in "ab":
            holds[f"sig{rd}_weak{wk}"] = res[rd]["nofear_ABC"] and res[rd]["TF_sig_in_D"] and weaker[wk] and AB
    if verbose:
        print("significance reading a (any interval p < 0.005):", {f"{c}:{g}": v for (c, g), v in sig_a.items()})
        print("significance reading b (interval effect p < 0.05):", {f"{c}:{g}": v for (c, g), v in sig_b.items()})
        print("D beta first interval %.3f vs E %.3f; largest D %.3f vs E %.3f" % (
            Bs["D"][:, 0, 0].mean(), Bs["E"][:, 0, 0].mean(), Bs["D"][:, :, 0].mean(0).max(), Bs["E"][:, :, 0].mean(0).max()))
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in out.items()})
    return dict(**{k: v for k, v in out.items() if k.startswith("p_")}, AB_ok=AB, **holds,
                holds=any(holds.values()))


combo_check(ONSET, PRIMARY, verbose=True)
r5s1 = show_readings(lambda r: combo_check(ONSET, r))

**Result: reproduced, resting on the trial baseline (assumed).** Under the primary reading no fear regressor reaches significance in A, B or C under either significance reading. Total fear alone (D) does (interval effect p = 0.0087, and a beta above zero at p < 0.005 in at least one interval). Its first-interval and largest mean beta (0.41) is below that of probability alone (0.87). A and B show a main effect of regressor (p = 0.015 and 0.004) with probability above fear. 9 of 18 readings of the amended claims file meet the criterion: the nine with the trial baseline, under both significance readings and both readings of "weaker". With the type baseline A's regressor effect is p = 0.060 to 0.12 and the claim fails, whatever the windows. Under the former 2 s readings, significance reading b fails for D (interval effect p 0.40 to 0.91).

### fig5e-onset-tuning-curve

> Varying the probability assigned to uncertainty from 0 to 1 in 0.125 steps, the Onset
> population's mean probability beta in the first interval peaks at the actual 0.375, with the
> flanking values next highest and betas falling off toward the extremes.

Criterion: nine regressions per neuron with all three regressors; mean probability beta over
the 29 neurons per assignment. Peak at 0.375 and the next two at 0.25 and 0.5; partially when
the peak holds and the order of the next two differs.

In [ ]:
PU = np.arange(0, 1.0001, 0.125)


def tuning(ks, r, iv):
    return np.array([betas(ks, r, [iv], prob=tuning_prob(pu))[:, 0, 2].mean() for pu in PU])


def check_tuning(ks, iv, r, verbose=False):
    t = tuning(ks, r, iv)
    order = PU[np.argsort(-t)]
    if verbose:
        print("mean probability beta by uncertainty assignment:", dict(zip(PU.tolist(), np.round(t, 3).tolist())))
    return dict(peak=order[0], second=order[1], third=order[2],
                holds=(order[0] == 0.375) & (set(order[1:3]) == {0.25, 0.5}))


check_tuning(ONSET, CUE_IV[0], PRIMARY, verbose=True)
r5e = show_readings(lambda r: check_tuning(ONSET, CUE_IV[0], r))

**Result: reproduced, resting on the primary reading (assumed).** The mean probability beta peaks at 0.375 (0.863), with 0.25 (0.838) and 0.5 (0.808) next, falling to 0.675 at 0 and 0.133 at 1. The peak is at 0.375 under all 18 readings of the amended claims file, and 14 of them meet the criterion. The four that do not use the type baseline with 0/0 set to 0 or left out under readings a and b of detail 1, where the third value is 0.125 in place of 0.5 (partially reproduced there). Under the former 2 s readings with 0/0 set to 1 or 0, the peak was at 0.25.

### fig5s2-binary-regressor

> Onset firing is better captured by the actual shock probability (uncertainty 0.375) than by
> a binary regressor equating uncertainty with safety, in the first interval and across the cue.

Criterion: a second set of regressions with the binary regressor (danger 1, uncertainty and
safety 0) in place of probability; repeated-measures ANOVA on betas with analysis, regressor and
interval. Reproduced when the analysis x regressor x interval interaction (all ten intervals)
and the analysis x regressor interaction (first interval) have p < 0.05 and the first-interval
mean beta is larger for probability than for binary.

In [ ]:
def check5s2(r, verbose=False):
    Bp = betas(ONSET, r)
    Bb = betas(ONSET, r, prob=tuning_prob(0.0))
    Y = np.stack([Bp, Bb], 1).transpose(0, 1, 3, 2)          # n x analysis x regressor x interval
    an = pag.rm_anova(Y, ("analysis", "regressor", "interval"))
    an1 = pag.rm_anova(Y[:, :, :, 0], ("analysis", "regressor"))
    if verbose:
        print("all intervals:", fmt_anova(an))
        print("first interval:", fmt_anova(an1))
        print("first-interval mean beta: probability %.3f, binary %.3f" % (Bp[:, 0, 2].mean(), Bb[:, 0, 2].mean()))
    holds = (an["analysisxregressorxinterval"][3] < 0.05) & (an1["analysisxregressor"][3] < 0.05) & (Bp[:, 0, 2].mean() > Bb[:, 0, 2].mean())
    return dict(F3=an["analysisxregressorxinterval"][0], p3=an["analysisxregressorxinterval"][3],
                F2=an1["analysisxregressor"][0], p2=an1["analysisxregressor"][3],
                P=Bp[:, 0, 2].mean(), binary=Bb[:, 0, 2].mean(), holds=holds)


check5s2(PRIMARY, verbose=True)
r5s2 = show_readings(check5s2)

**Result: reproduced.** The analysis x regressor x interval interaction is F(18,504) = 2.49, p = 6.4e-4 (paper 2.51, 5.89e-4). The first-interval analysis x regressor interaction is F(2,56) = 7.75, p = 0.0011 (paper 7.80, 1.03e-3). The first-interval beta is 0.863 for probability against 0.675 for binary. All 18 readings of the amended claims file meet the criterion (largest p 0.049 for the three-way and 0.039 for the first-interval interaction). Three of the former 2 s readings miss one interaction at p < 0.05.

### fig5f-ramping-last-interval

> In the last 1 s cue interval, Ramping probability and total fear betas both exceed the
> interval fear beta and do not differ from each other.

Criterion: last-interval betas of the 14 Ramping neurons; both comparisons with interval fear
significant (p < 0.05) in the stated direction, probability against total fear p >= 0.05.

In [ ]:
def check5f(r, verbose=False):
    B = betas(RAMP, r, CUE_IV[9:])[:, 0]
    IF, TF, P = B.T
    a, b, c = stats.ttest_rel(P, IF), stats.ttest_rel(TF, IF), stats.ttest_rel(P, TF)
    if verbose:
        print(f"mean betas IF {IF.mean():.3f}, TF {TF.mean():.3f}, P {P.mean():.3f}")
        print(f"P vs IF t(13) = {a.statistic:.2f}, p = {a.pvalue:.3g}; TF vs IF t = {b.statistic:.2f}, p = {b.pvalue:.3g}; P vs TF t = {c.statistic:.2f}, p = {c.pvalue:.3g}")
    holds = (a.statistic > 0) & (a.pvalue < 0.05) & (b.statistic > 0) & (b.pvalue < 0.05) & (c.pvalue >= 0.05)
    return dict(IF=IF.mean(), TF=TF.mean(), P=P.mean(), p_PvIF=a.pvalue, p_TFvIF=b.pvalue, p_PvTF=c.pvalue, holds=holds)


check5f(PRIMARY, verbose=True)
r5f = show_readings(check5f)

**Result: reproduced.** Last-interval betas over the 14 Ramping neurons are probability 0.777, total fear 0.478 and interval fear -0.064. Probability (t(13) = 3.62, p = 0.0031) and total fear (t = 4.62, p = 4.8e-4) exceed interval fear, and they do not differ from each other (t = 1.14, p = 0.28). The paper reports 3.61, 4.68 and 1.11. All 18 readings of the amended claims file meet the criterion (total fear beta 0.41 to 0.84). Under the former 2 s readings with 0/0 set to 1 or 0, the total fear beta falls to 0.12 or less and the claim fails there.

### fig5gh-ramping-beta-bias

> Ramping single-unit betas in the last interval are biased positive for probability and total
> fear, which are uncorrelated; they are not biased for interval fear, and the probability and
> interval fear betas are negatively correlated.

Criterion: sign tests against zero (two-sided, p < 0.05) and Pearson correlations (p < 0.05)
across the 14 neurons. Four parts: P and TF biased positive; their correlation p >= 0.05; IF not
biased; P and IF correlate negatively. Partially when three of four hold.

In [ ]:
def check5gh(r):
    B = betas(RAMP, r, CUE_IV[9:])[:, 0]
    IF, TF, P = B.T
    sP, sT, sI = pag.sign_test(P), pag.sign_test(TF), pag.sign_test(IF)
    cPT, cPI = stats.pearsonr(P, TF), stats.pearsonr(P, IF)
    parts = [(sP[2] < 0.05) & (sP[0] > sP[1] / 2) & (sT[2] < 0.05) & (sT[0] > sT[1] / 2), cPT[1] >= 0.05,
             sI[2] >= 0.05, (cPI[0] < 0) & (cPI[1] < 0.05)]
    return dict(P_pos=f"{sP[0]}/{sP[1]}", p_P=sP[2], TF_pos=f"{sT[0]}/{sT[1]}", p_TF=sT[2], IF_pos=f"{sI[0]}/{sI[1]}",
                p_IF=sI[2], r_PTF=cPT[0], p_rPTF=cPT[1], r_PIF=cPI[0], p_rPIF=cPI[1], parts_held=int(sum(parts)),
                holds=all(parts))


r5gh = show_readings(check5gh)

**Result: reproduced, resting on the primary reading (assumed).** Under the primary reading probability and total fear are each biased positive (12 of 14, p = 0.013) and uncorrelated (r = -0.34, p = 0.24). Interval fear is not biased (6 of 14, p = 0.79), and probability and interval fear correlate negatively (r = -0.57, p = 0.035). 9 of 18 readings of the amended claims file meet all four parts: all six of reading c of detail 1 and three of reading b. Reading a meets two or three parts. Of the former readings, 3 of 12 meet all four.

### fig5i-ramping-all-intervals

> Over all ten cue intervals Ramping neurons prioritize probability (mean beta 0.58, against
> 0.12 for total fear and -0.03 for interval fear); only the probability beta differs from zero
> and it exceeds both fear betas, with a main effect of regressor and a regressor x interval
> interaction.

Criterion: per neuron, the mean of its betas over the ten intervals; means within 0.58 +- 0.14,
0.12 +- 0.10, -0.03 +- 0.05 under one reading; ANOVA effects p < 0.01; the probability mean
differs from zero and exceeds both fear means at p < 0.05; the fear means do not differ from
zero (p >= 0.05). Partially when the tests hold and a mean is outside tolerance.

In [ ]:
def check5i(r, verbose=False):
    B = betas(RAMP, r)
    M = B.mean(1)
    IF, TF, P = M.T
    an = pag.rm_anova(B.transpose(0, 2, 1), ("regressor", "interval"))
    zP, zT, zI = onesamp(P), onesamp(TF), onesamp(IF)
    a, b = stats.ttest_rel(P, TF), stats.ttest_rel(P, IF)
    means = (abs(P.mean() - 0.58) <= 0.14) & (abs(TF.mean() - 0.12) <= 0.10) & (abs(IF.mean() + 0.03) <= 0.05)
    tests = (an["regressor"][3] < 0.01) & (an["regressorxinterval"][3] < 0.01) & (zP.pvalue < 0.05) & (zP.statistic > 0) \
        & (a.statistic > 0) & (a.pvalue < 0.05) & (b.statistic > 0) & (b.pvalue < 0.05) & (zT.pvalue >= 0.05) & (zI.pvalue >= 0.05)
    if verbose:
        print(fmt_anova(an))
        print(f"mean over intervals: P {P.mean():.3f} +- {P.std(ddof=1) / np.sqrt(len(P)):.3f}, TF {TF.mean():.3f} +- {TF.std(ddof=1) / np.sqrt(len(P)):.3f}, "
              f"IF {IF.mean():.3f} +- {IF.std(ddof=1) / np.sqrt(len(P)):.3f}")
        print(f"against zero: P t(13) = {zP.statistic:.2f}, p = {zP.pvalue:.3g}; TF t = {zT.statistic:.2f}, p = {zT.pvalue:.3g}; IF t = {zI.statistic:.2f}, p = {zI.pvalue:.3g}")
        print(f"P vs TF t = {a.statistic:.2f}, p = {a.pvalue:.3g}; P vs IF t = {b.statistic:.2f}, p = {b.pvalue:.3g}")
    return dict(P=P.mean(), TF=TF.mean(), IF=IF.mean(), p_reg=an["regressor"][3], p_int=an["regressorxinterval"][3],
                p_P0=zP.pvalue, p_TF0=zT.pvalue, p_IF0=zI.pvalue, p_PvTF=a.pvalue, p_PvIF=b.pvalue,
                means_ok=means, tests_ok=tests, holds=means & tests)


check5i(PRIMARY, verbose=True)
r5i = show_readings(check5i)

**Result: reproduced.** Under the primary reading the per-neuron means over the ten intervals are probability 0.577 +- 0.136, total fear 0.120 +- 0.100 and interval fear -0.026 +- 0.050 (paper 0.58 +- 0.14, 0.12 +- 0.10, -0.03 +- 0.05). The regressor effect is F(2,26) = 9.14 (p = 9.9e-4) and the interaction F(18,234) = 2.19 (p = 0.0042). Probability differs from zero (t(13) = 4.24, p = 9.6e-4) and exceeds total fear (t = 2.35, p = 0.035) and interval fear (t = 4.37, p = 7.5e-4). The fear means do not differ from zero (p = 0.25, 0.61). The paper reports t = 4.21, 2.32 and 4.32. 7 of 18 readings of the amended claims file meet the whole criterion. The first of them in the order the claims file lists the readings (2 s / 10 s, 0/0 left out, trial baseline) gives means of 0.686, 0.045 and -0.062. The other 11 are the nine type-baseline readings, which fail a test, and the readings with 0/0 set to 0 under a and b of detail 1, which put the interval fear mean outside tolerance (-0.095 and -0.096).

### fig5s1-ramping-regressor-combinations

> For Ramping neurons, the prioritization of probability over all ten intervals holds when
> probability is compared with total fear or interval fear alone, and when each regressor is
> used in isolation.

Criterion: regressions F (P + TF), G (P + IF), H (IF), I (TF), J (P); per neuron the mean beta
over the ten intervals. In F and G the probability mean differs from zero (p < 0.05), the fear
mean does not, probability exceeds fear (paired, p < 0.05), and the ANOVA (regressor x
interval) has a main effect of regressor (p < 0.05); of H, I and J only J's mean differs from
zero. Partially when F and G hold and H to J do not.

In [ ]:
def check5s1r(r, verbose=False):
    Bs = {c: betas(RAMP, r, regs=g) for c, g in COMBOS.items()}
    fg = []
    out = {}
    for c, name in (("A", "F"), ("B", "G")):
        M = Bs[c].mean(1)
        P, Fe = M[:, 0], M[:, 1]
        an = pag.rm_anova(Bs[c].transpose(0, 2, 1), ("regressor", "interval"))
        zP, zF, d = onesamp(P), onesamp(Fe), stats.ttest_rel(P, Fe)
        ok = (zP.pvalue < 0.05) & (zP.statistic > 0) & (zF.pvalue >= 0.05) & (d.statistic > 0) & (d.pvalue < 0.05) & (an["regressor"][3] < 0.05)
        out.update({f"{name}_P": P.mean(), f"{name}_fear": Fe.mean(), f"{name}_pP0": zP.pvalue, f"{name}_pF0": zF.pvalue,
                    f"{name}_pdiff": d.pvalue, f"{name}_preg": an["regressor"][3]})
        fg.append(ok)
    single = {}
    for c, name in (("C", "H"), ("D", "I"), ("E", "J")):
        m = Bs[c].mean(1)[:, 0]
        z = onesamp(m)
        an = pag.rm_anova(Bs[c][:, :, 0])["A"]
        single[name] = z.pvalue < 0.05
        out.update({f"{name}_mean": m.mean(), f"{name}_p0": z.pvalue, f"{name}_F_int": an[0], f"{name}_p_int": an[3]})
    hij = single["J"] and not single["H"] and not single["I"]
    if verbose:
        print({k: round(float(v), 4) for k, v in out.items()})
    return dict(**out, FG_ok=all(fg), HIJ_ok=hij, holds=all(fg) and hij)


check5s1r(PRIMARY, verbose=True)
r5s1r = show_readings(check5s1r)

**Result: partially reproduced, resting on the trial baseline (assumed).** Under the primary reading F and G hold: probability means 0.58 and 0.66 differ from zero (p = 9.3e-4, 7.5e-5), fear means do not (p = 0.41, 0.92), probability exceeds fear (p = 0.029, 4.2e-5), and the regressor effects are p = 0.029 and 4.2e-5. Of the single-regressor models, probability alone (J, 0.66, p = 1.7e-4) differs from zero, and so does total fear alone (I, 0.37, p = 0.0067). Interval fear alone (H) does not (p = 0.14). Total fear alone differs from zero under all 18 readings of the amended claims file (0.31 to 0.41, p 0.0014 to 0.0067), so no reading meets the criterion in full. F and G hold in 12 readings, those with the trial baseline or with reading a of detail 1. In the six type-baseline readings under b and c, F's comparison of probability with total fear has p = 0.051, and the claim is not reproduced there. Under the six former readings with 2 s for both ratios, total fear alone was near zero (p 0.24 to 0.92) and the claim held in full in all six; it held in none of the six former 10 s readings, so in 6 of the 12 former readings.

### fig5j-ramping-tuning-curve

> The Ramping population's tuning curve for the last interval is shifted right of the actual
> probability, peaking at 0.625, consistent with mixed signaling of probability and fear.

Criterion: as fig5e on the 14 Ramping neurons and the last interval; peak at 0.625; partially
when at 0.5 or 0.75.

In [ ]:
check_tuning(RAMP, CUE_IV[9], PRIMARY, verbose=True)
r5j = show_readings(lambda r: dict(peak=PU[np.argmax(tuning(RAMP, r, CUE_IV[9]))],
                                   holds=PU[np.argmax(tuning(RAMP, r, CUE_IV[9]))] == 0.625))

**Result: reproduced.** The Ramping last-interval tuning curve peaks at 0.625 (0.866), just above 0.5 (0.858). All 18 readings of the amended claims file peak at 0.625. Under four former 2 s readings the peak was at 0.5.

## After cue offset

### fig6ab-post-cue-firing

> In the 5 s after cue offset (delay, shock, post-shock; the shock itself excluded) Onset
> neurons show no effects of trial type or bin, while Ramping neurons keep firing to danger and
> uncertainty through the delay, with a trial x bin interaction, and the two populations differ.

Criterion: Z-scored 100 ms firing for the four trial types, 45 bins from cue offset to 5 s
after it without the shock (2.0 to 2.5 s after offset). Onset: no effect at p < 0.05; Ramping:
both main effects and the interaction at p < 0.01; mixed ANOVA with neuron type: the three
neuron-type interactions at p < 0.01. Partially when two of three parts hold.

In [ ]:
POST_BINS = [i for i in range(200, 250) if not 220 <= i < 225]
Z100P = {k: pag.population_z100(units[k], split_uncertainty=True) for k in ONSET + RAMP}


def zpost(ks):
    return np.array([[Z100P[k][t][POST_BINS] for t in pag.TYPES] for k in ks])


YO, YRp = zpost(ONSET), zpost(RAMP)
print("bins:", len(POST_BINS))
print("Onset:", fmt_anova(pag.rm_anova(YO, ("trial", "bin"))))
print("Ramping:", fmt_anova(pag.rm_anova(YRp, ("trial", "bin"))))
mx = pag.mixed_anova([YO, YRp])
print("mixed (G = neuron type, A = trial type, B = bin):", fmt_anova(mx))
print("Ramping mean Z over the delay (0 to 2 s after offset): " + ", ".join(f"{t} {YRp[:, i, :20].mean():.2f}" for i, t in enumerate(pag.TYPES)))
print("Onset mean Z over the delay: " + ", ".join(f"{t} {YO[:, i, :20].mean():.2f}" for i, t in enumerate(pag.TYPES)))

In [ ]:
# check of pag.mixed_anova on synthetic data against sums of squares by model comparison
# (least squares on indicator columns, terms entered in order, each tested against its error term)
rng_mx = np.random.default_rng(13)
G_ = [rng_mx.normal(size=(5, 3, 4)) + rng_mx.normal(size=(1, 3, 4)), rng_mx.normal(size=(8, 3, 4))]
Y_ = np.concatenate(G_)
s_, a_, b_ = [x.ravel() for x in np.meshgrid(np.arange(len(Y_)), np.arange(3), np.arange(4), indexing="ij")]
g_ = np.repeat([0, 1], [5, 8])[s_]
y_ = Y_.ravel()


def indicators(*cols):
    key = np.zeros(len(y_), int)
    for c in cols:
        key = key * (c.max() + 1) + c
    return np.eye(np.unique(key).size)[np.unique(key, return_inverse=True)[1]]


TERMS = [("G", (g_,)), ("S", (s_,)), ("A", (a_,)), ("AxG", (a_, g_)), ("AxS", (a_, s_)), ("B", (b_,)),
         ("BxG", (b_, g_)), ("BxS", (b_, s_)), ("AxB", (a_, b_)), ("AxBxG", (a_, b_, g_))]
cols, rss, SS = [np.ones(len(y_))], [], {}
for name, cs in TERMS:
    cols.append(indicators(*cs))
    X_ = np.column_stack(cols)
    r_ = y_ - X_ @ np.linalg.lstsq(X_, y_, rcond=None)[0]
    rss.append(r_ @ r_)
    SS[name] = (rss[-2] if len(rss) > 1 else ((y_ - y_.mean()) ** 2).sum()) - rss[-1]
SS["AxBxS"] = rss[-1]
n_, k_ = len(Y_), 2
DF = {"G": 1, "S": n_ - k_, "A": 2, "AxG": 2, "AxS": 2 * (n_ - k_), "B": 3, "BxG": 3, "BxS": 3 * (n_ - k_),
      "AxB": 6, "AxBxG": 6, "AxBxS": 6 * (n_ - k_)}
ERR = {"G": "S", "A": "AxS", "AxG": "AxS", "B": "BxS", "BxG": "BxS", "AxB": "AxBxS", "AxBxG": "AxBxS"}
ours = pag.mixed_anova(G_)
for t, e in ERR.items():
    F_ref = (SS[t] / DF[t]) / (SS[e] / DF[e])
    print(f"{t}: reference F = {F_ref:.6f}, pag.mixed_anova F = {ours[t][0]:.6f}")


**Result: reproduced.** Onset neurons show no effect after cue offset: trial F(3,84) = 2.23, p = 0.091; bin p = 0.14; interaction p = 0.91 (paper Fs < 2.3, ps > 0.09). Ramping neurons show trial F(3,39) = 14.14, bin F(44,572) = 7.11, and trial x bin F(132,1716) = 3.14 (paper 3.14), all p < 1e-5. In the mixed ANOVA the neuron type interactions are trial F(3,123) = 5.70 (p = 0.0011), bin F(44,1804) = 4.17 and trial x bin F(132,5412) = 2.56, all p < 0.01. On synthetic data, every F of pag.mixed_anova equals that of sums of squares by model comparison (cell above). Ramping firing stays high through the delay on danger and uncertainty trials (Z 1.16 to 1.40) and not on safety trials (-0.10).

### fig6c-onset-post-cue-regression

> After cue offset Onset neurons signal neither interval fear nor probability; no effect of
> regressor or regressor x interval interaction, and no beta differs from zero.

Criterion (absence): regressions per neuron in the nine 500 ms post-cue intervals (four of the
delay, five after the shock) on interval fear (that 500 ms interval's suppression ratio) and
probability; repeated-measures ANOVA at p < 0.05 and single-sample t-tests against zero at p <
0.005. Reproduced when neither ANOVA effect is significant and no beta differs from zero.

In [ ]:
def post_check(ks, r, verbose=False):
    B = betas(ks, r, POST_IV, regs=("IF", "P"))       # n x 9 x 2
    an = pag.rm_anova(B.transpose(0, 2, 1), ("regressor", "interval"))
    p0 = np.array([[onesamp(B[:, j, i]).pvalue for j in range(9)] for i in range(2)])
    t0 = np.array([[onesamp(B[:, j, i]).statistic for j in range(9)] for i in range(2)])
    if verbose:
        print(fmt_anova(an))
        print("mean probability beta by interval:", np.round(B[:, :, 1].mean(0), 3).tolist())
        print("p against zero, probability:", [f"{p:.2g}" for p in p0[1]])
        print("p against zero, interval fear:", [f"{p:.2g}" for p in p0[0]])
    return B, an, p0, t0


def check6c(r, verbose=False):
    B, an, p0, t0 = post_check(ONSET, r, verbose)
    holds = (an["regressor"][3] >= 0.05) & (an["regressorxinterval"][3] >= 0.05) & (p0.min() >= 0.005)
    return dict(F_reg=an["regressor"][0], p_reg=an["regressor"][3], F_int=an["regressorxinterval"][0],
                p_int=an["regressorxinterval"][3], min_p0=p0.min(), holds=holds)


check6c(PRIMARY, verbose=True)
r6c = show_readings(check6c)

**Result: reproduced.** Onset regressions after the cue show no regressor effect (F(1,28) = 0.51, p = 0.48; paper 1.02, 0.32) and no interaction (F(8,224) = 0.61, p = 0.77; paper 0.53, 0.83). No beta differs from zero at p < 0.005 (smallest p 0.077). All 18 readings of the amended claims file give the same (smallest p 0.0064 to 0.09), as do the 12 former readings.

### fig6d-ramping-post-cue-regression

> Ramping neurons signal probability throughout the delay (the first four 500 ms intervals
> after cue offset), and this signal diminishes after shock delivery, with a regressor x
> interval interaction.

Criterion: as fig6c on the 14 Ramping neurons. Reproduced when the interaction has p < 0.01,
the mean probability beta is above zero at p < 0.005 in all four delay intervals, and the mean
over the five post-shock intervals is below the mean over the delay; partially when the
interaction holds and the probability beta is significant in one to three delay intervals.

In [ ]:
def check6d(r, verbose=False):
    B, an, p0, t0 = post_check(RAMP, r, verbose)
    nsig = int(((p0[1, :4] < 0.005) & (t0[1, :4] > 0)).sum())
    delay, post = B[:, :4, 1].mean(), B[:, 4:, 1].mean()
    holds = (an["regressorxinterval"][3] < 0.01) & (nsig == 4) & (post < delay)
    return dict(F_int=an["regressorxinterval"][0], p_int=an["regressorxinterval"][3], delay_sig_of_4=nsig,
                delay_mean=delay, post_mean=post, holds=holds,
                partial=(an["regressorxinterval"][3] < 0.01) & (1 <= nsig <= 3))


check6d(PRIMARY, verbose=True)
r6d = show_readings(check6d)

**Result: reproduced.** For Ramping neurons the regressor x interval interaction is F(8,104) = 7.00, p = 2.3e-7 (paper 5.20). The probability beta is above zero in all four delay intervals (p from 2.7e-6 to 7.7e-4). Its mean falls from 0.715 over the delay to 0.020 over the five post-shock intervals. All 18 readings of the amended claims file and the 12 former readings give the same.

### fig6ef-beta-correlations

> Probability betas of the delay intervals correlate with those of the post-shock intervals in
> 13 of 20 comparisons for Onset neurons but 4 of 20 for Ramping neurons, a significant
> difference (6 and 1 of 20 after Bonferroni correction, also different).

Criterion: per population, Pearson correlation across neurons of the probability beta of each
delay interval with each post-shock interval (20 pairs), significant at p < 0.05 and at p <
0.01. Chi-square on the 2 x 2 counts, readings (a) Pearson chi-square x (N - 1) / N, (b)
Pearson, (c) Yates. Reproduced when, under one reading, the chi-square has p < 0.05 at both
thresholds and the four counts are within tolerance (13 +- 1.3, 4 +- 0.4, 6 +- 0.6, 1 +- 0.1);
partially when Onset has more significant comparisons at both thresholds but a count or a test
differs.

In [ ]:
def corr_counts(ks, r):
    B = betas(ks, r, POST_IV, regs=("IF", "P"))[:, :, 1]
    ps = np.array([[stats.pearsonr(B[:, i], B[:, j])[1] for j in range(4, 9)] for i in range(4)])
    return ps


def chi(a, b, n=20):
    tab = np.array([[a, n - a], [b, n - b]])
    out = {}
    if (tab.sum(0) == 0).any():
        return {"a": np.nan, "b": np.nan, "c": np.nan}, {"a": np.nan, "b": np.nan, "c": np.nan}
    c2 = stats.chi2_contingency(tab, correction=False)[0]
    N = tab.sum()
    out = {"a": c2 * (N - 1) / N, "b": c2, "c": stats.chi2_contingency(tab, correction=True)[0]}
    return out, {k: stats.chi2.sf(v, 1) for k, v in out.items()}


def check6ef(r, verbose=False):
    po, pr = corr_counts(ONSET, r), corr_counts(RAMP, r)
    res = {}
    for thr in (0.05, 0.01):
        a, b = int((po < thr).sum()), int((pr < thr).sum())
        c, p = chi(a, b)
        res[thr] = (a, b, c, p)
    counts_ok = (abs(res[0.05][0] - 13) <= 1.3) & (abs(res[0.05][1] - 4) <= 0.4) & (abs(res[0.01][0] - 6) <= 0.6) & (abs(res[0.01][1] - 1) <= 0.1)
    tests_ok = {k: (res[0.05][3][k] < 0.05) & (res[0.01][3][k] < 0.05) for k in "abc"}
    if verbose:
        print("Onset p (rows: delay intervals 1-4, columns: post-shock 6-10):\n", np.round(po, 3))
        print("Ramping p:\n", np.round(pr, 3))
        for thr, (a, b, c, p) in res.items():
            print(f"p < {thr}: Onset {a}/20, Ramping {b}/20; chi-square " + ", ".join(f"({k}) {c[k]:.2f}, p = {p[k]:.3g}" for k in "abc"))
    return dict(on05=res[0.05][0], ra05=res[0.05][1], on01=res[0.01][0], ra01=res[0.01][1],
                chi05_a=res[0.05][2]["a"], p05_a=res[0.05][3]["a"], chi01_a=res[0.01][2]["a"], p01_a=res[0.01][3]["a"],
                counts_ok=counts_ok, tests_ok_any=any(tests_ok.values()), holds=counts_ok & any(tests_ok.values()),
                partial=(res[0.05][0] > res[0.05][1]) & (res[0.01][0] > res[0.01][1]))


check6ef(PRIMARY, verbose=True)
r6ef = show_readings(check6ef)

**Result: partially reproduced, resting on the primary reading (assumed).** Onset has more significant delay against post-shock correlations than Ramping at both thresholds, but far fewer than the paper reports: 3 of 20 against 0 at p < 0.05 (paper 13 and 4), and 1 against 0 at p < 0.01 (paper 6 and 1). No chi-square reading is significant (reading a 3.16, p = 0.075, and 1.00, p = 0.32). Across the 18 readings of the amended claims file the Onset count at p < 0.05 is 1 to 5, and no reading meets the criterion in full. Two readings (0/0 left out, type baseline, under readings a and b of detail 1) give Ramping more (2 against 1), which would be not reproduced. No former reading meets the criterion. The outcome is partial only through the criterion's partial branch (Onset more than Ramping at both thresholds, 3 > 0 and 1 > 0); the reported counts are missed by a factor of about four under every reading.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1c-total-suppression | reproduced |
| fig1c-nonlinear-scaling | reproduced |
| fig1d-interval-suppression | reproduced (assumed reading) |
| fig1s1a-nose-poke-rates | partially reproduced (assumed reading) |
| fig1s1b-sr-poke-correlation | reproduced |
| fig1e-two-clusters | reproduced |
| onset-population | reproduced |
| ramping-population | reproduced |
| fig2a-onset-anova | reproduced |
| fig2b-onset-first-interval | reproduced |
| fig2b-onset-last-interval | reproduced |
| fig2c-ramping-anova | reproduced |
| fig2d-ramping-first-interval | reproduced |
| fig2d-ramping-last-interval | reproduced |
| fig2e-departure-times | partially reproduced |
| fig2e-rate-no-difference | reproduced |
| fig2s1-single-trial | partially reproduced |
| fig2s2-nose-poke-cessation | reproduced |
| fig3-onset-ramping-independent | partially reproduced |
| fig4abc-onset-single-unit-biases | reproduced |
| fig4d-onset-patterns | reproduced |
| fig4efg-ramping-single-unit-biases | partially reproduced |
| fig4h-ramping-patterns | reproduced |
| fig5a-onset-probability-first-interval | reproduced |
| fig5bc-onset-beta-bias | reproduced (assumed reading) |
| fig5d-onset-all-intervals | reproduced (assumed reading) |
| fig5s1-regressor-combinations | reproduced (assumed reading) |
| fig5e-onset-tuning-curve | reproduced (assumed reading) |
| fig5s2-binary-regressor | reproduced |
| fig5f-ramping-last-interval | reproduced |
| fig5gh-ramping-beta-bias | reproduced (assumed reading) |
| fig5i-ramping-all-intervals | reproduced |
| fig5s1-ramping-regressor-combinations | partially reproduced (assumed reading) |
| fig5j-ramping-tuning-curve | reproduced |
| fig6ab-post-cue-firing | reproduced |
| fig6c-onset-post-cue-regression | reproduced |
| fig6d-ramping-post-cue-regression | reproduced |
| fig6ef-beta-correlations | partially reproduced (assumed reading) |
| fig1s5-hfr-units | reproduced |

How the analysis differs from the paper's. The paper states neither nose poke baseline nor
the 0/0 rule. The primary reading is the one the paper's printed regression input reproduces: 2 s
for interval fear, 20 s for total fear, 0/0 = 1. The claims file, amended after this attempt's
question, lists it among its readings (reading b of detail 1, reading c of detail 2), and every
reading of the amended file is run. Where they give different outcomes, the outcome rests on the
primary reading and is marked as assumed. The claims file's earlier readings (one window for
both ratios) are shown as "former" and do not count. The repeated-measures ANOVAs carry no
sphericity correction, as the paper's integer degrees of freedom imply. k-means uses k-means++
starts and 20 replicates. The waveform features use the peak-picking described in pag.py.
Interval Z firing is Z-scored per interval, as the Methods describe.

What was found about the dataset:

- The files hold everything the paper analysed: 245 units, 88 sessions, 6 rats, 32 trials each,
  with spike times, cue, shock and poke times, and mean waveforms. With the paper's methods
  they reproduce the population counts (29, 17, 3, 14). Few statistics match to their printed
  precision (F = 7.16 in fig5d, F = 3.14 in fig6ab, the fig5i means). Many are close but not
  exact (fig1c F = 595 against 592.00, fig2b t = 5.05 against 4.54, fig6d F = 7.00 against
  5.20), and some miss widely (fig6ef 3 of 20 against 13 of 20, the fig2e departure times).
- The data description says the cube's `poke.base` is the poke rate 2 s before the cue. The
  data set's cubeScript.m computes it over 20 s, and sets 0/0 suppression ratios to 1. Neither
  the paper nor the description states either choice; the paper's table implies both.
- The cube's total and interval suppression ratios (cubeScript.m, lines printed in the data
  check) use the 20 s baseline, while the
  paper's interval fear, by its table and by Figure 1D's statistics, uses 2 s.
- Cue onsets are 132 to 195 s apart, so no ITI reaches the "mean ITI of 3.5 min" of the paper
  and the description.
- Five unit pairs recorded in the same session share 66 to 90% of the spikes of one unit
  (within 0.5 ms); one is an Onset neuron.
- Only three Onset-Ramping pairs share a session, where the paper reports four. Nine units
  just above the screen's threshold (p 0.025 to 0.046) would each add one.
- The spike trains hold spikes only from 20 s before each cue onset to 31 to 34 s after it,
  with no spikes between these windows. The description does not say so, and a rate over the
  span of the train is about a third of the firing rate.
- Interval Z firing ties each neuron's cue means together, so the cross-neuron correlations of
  Figure 4 are negative even with the trial labels shuffled; the observed correlations are within
  the shuffled range.

In [ ]:
print(f"run time {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")